# 1 Exploração dos Dados

&nbsp;&nbsp;&nbsp;&nbsp; Nesta etapa, foi realizada uma análise exploratória detalhada dos dados, com o objetivo de compreender as principais características das variáveis e suas relações. O processo seguiu as seguintes etapas:

### 1.1 Importação de Bibliotecas Necessárias

&nbsp;&nbsp;&nbsp;&nbsp; No código abaixo, são importadas as principais bibliotecas utilizadas para o tratamento de dados e a criação de gráficos, como Numpy, Pandas, Matplotlib, Seaborn e Scikit-Learn.

In [ ]:
# Instalação de streamlit e outras bibliotecas para o funcionamento momentâneo do notebook
!pip install numpy pandas matplotlib seaborn scikit-learn lightgbm shap

Defaulting to user installation because normal site-packages is not writeable
  Using cached lightgbm-4.6.0-py3-none-win_amd64.whl.metadata (17 kB)
  Using cached shap-0.48.0-cp313-cp313-win_amd64.whl.metadata (25 kB)
  Using cached pycaret-3.3.2-py3-none-any.whl.metadata (17 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached slicer-0.0.8-py3-none-any.whl.metadata (4.0 kB)
  Using cached numba-0.62.1-cp313-cp313-win_amd64.whl.metadata (2.9 kB)
  Using cached cloudpickle-3.1.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached ipywidgets-8.1.7-py3-none-any.whl.metadata (2.4 kB)
  Using cached numpy-1.26.4-cp313-cp313-win_amd64.whl
  Using cached pandas-2.1.4.tar.gz (4.3 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'

  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 2
  ╰─> [98 lines of output]
      + meson setup C:\Users\Inteli\AppData\Local\Temp\pip-install-twctza9j\pandas_863a8453886147e28d371eb1bc0a0c55 C:\Users\Inteli\AppData\Local\Temp\pip-install-twctza9j\pandas_863a8453886147e28d371eb1bc0a0c55\.mesonpy-z7f_drhk\build -Dbuildtype=release -Db_ndebug=if-release -Db_vscrt=md --vsenv --native-file=C:\Users\Inteli\AppData\Local\Temp\pip-install-twctza9j\pandas_863a8453886147e28d371eb1bc0a0c55\.mesonpy-z7f_drhk\build\meson-python-native-file.ini
      The Meson build system
      Version: 1.2.1
      Source dir: C:\Users\Inteli\AppData\Local\Temp\pip-install-twctza9j\pandas_863a8453886147e28d371eb1bc0a0c55
      Build dir: C:\Users\Inteli\AppData\Local\Temp\pip-install-twctza9j\pandas_863a8453886147e28d371eb1bc0a0c55\.mesonpy-z7f_drhk\build
      Build type: native build
      Project name: pandas
      Project version: 2.1.4

In [3]:
# Importar pacotes

# Manipulação e visualização de dados
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import inspect
import lightgbm as lgb

# Pré-processamento
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler, LabelEncoder
from sklearn.compose import ColumnTransformer

# Divisão de dados e seleção de modelos
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV

# Modelos
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.datasets import load_iris

# Métricas de avaliação
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, precision_recall_fscore_support,
    roc_curve, roc_auc_score, r2_score, mean_absolute_error, mean_squared_error
)

# Outros
from sklearn.utils.multiclass import unique_labels
from imblearn.over_sampling import SMOTE


ModuleNotFoundError: No module named 'lightgbm'

### 1.2 Importação das Planilhas de Dados

&nbsp;&nbsp;&nbsp;&nbsp; O código abaixo importa as planilhas disponibilizadas pela empresa parceira Finnet.

In [ ]:
# Carregar os arquivos CSV


# Carregamento em amostras para economia de memória e testes

df_gl = pd.read_csv("Grupo com registro entre 07-2024 a 06-2025- GL.csv", sep="\t", header=0, engine="python")
df_gm = pd.read_csv("Grupo com registro entre 07-2024 a 06-2025- GM.csv", sep="\t", header=0, engine="python")
df_gp = pd.read_csv("Grupo com registro entre 07-2024 a 06-2025- GP.csv", sep="\t", header=0, engine="python")
df_gf = pd.read_csv("Grupo com registro entre 07-2024 a 06-2025- GF.csv", sep="\t", header=0, engine="python")
df_gt = pd.read_csv("Grupo com registro entre 07-2024 a 06-2025- GT.csv", sep="\t", header=0, engine="python")
df_gv = pd.read_csv("Grupo com registro entre 07-2024 a 06-2025- GV.csv", sep="\t", header=0, engine="python")
df_gw = pd.read_csv("Grupo com registro entre 07-2024 a 06-2025- GW.csv", sep="\t", header=0, engine="python")

# Após carregar, conversão de floats para float32 para economizar memória também

for df in [df_gf, df_gl, df_gm, df_gp, df_gt, df_gv, df_gw]:
    for col in df.select_dtypes(include="float64").columns:
        df[col] = df[col].astype("float32")


 ### 1.3 Divisão de Colunas em Múltiplas Colunas

&nbsp;&nbsp;&nbsp;&nbsp; Após importar os arquivos CSV, todas as informações vieram concentradas em uma única coluna. Para resolver isso, foi criada uma função que divide corretamente os valores em múltiplas colunas, utiliza a primeira linha como cabeçalho e retorna o DataFrame ajustado. Essa função foi aplicada em todos os Dataframes.

In [ ]:
# Função para a divisão das colunas

def dividir_colunas(df):

    # Verifica se o DataFrame tem apenas uma coluna.

    if df.shape[1] == 1:

        # Divide a única coluna em múltiplas colunas.

        dados = df.iloc[:, 0].str.split('\t', expand=True)

        # Usa a primeira linha como nomes das colunas.

        dados.columns = dados.iloc[0]

        # Remove a primeira linha por ter virado cabeçalho.

        dados = dados.drop(index=0).reset_index(drop=True)
        return dados
    else:

        return df

df_gf = dividir_colunas(df_gf)
df_gl = dividir_colunas(df_gl)
df_gm = dividir_colunas(df_gm)
df_gp = dividir_colunas(df_gp)
df_gt = dividir_colunas(df_gt)
df_gv = dividir_colunas(df_gv)
df_gw = dividir_colunas(df_gw)

### 1.4 Conversão de Tipos de Dados


&nbsp;&nbsp;&nbsp;&nbsp; Esse código é responsável por padronizar os tipos de dados das colunas de diferentes DataFrames após a importação. Primeiro, ele organiza as colunas em quatro grupos: numéricas, categóricas, de datas e de texto. Em seguida, a função converter_tipos percorre cada grupo e faz os ajustes necessários. Para as colunas numéricas, substitui valores inválidos representados por \N por NaN e converte os dados para tipo numérico. Para as colunas categóricas, altera o tipo para category, o que facilita análises e economiza memória. Já nas colunas de datas, também substitui \N por NaN e converte para o formato datetime, permitindo operações temporais como filtragem e cálculos.

In [ ]:
# Divisão de tipos

colunas_numericas = ['multa','id_grupo', 'id_pagador', 'id_beneficiario', 'Numero_do_boleto', 'vl_boleto', 'vl_pagto', 'juros']
colunas_categoricas = ['tipo_multa', 'status_boleto', 'pagador_cep', 'banco', 'tipo_juros', 'pagador_cidade']
colunas_data = ['data_inclusao', 'data_vencto', 'dt_pagto', 'pagador_dt_ultimo_acesso ']
colunas_texto = [ 'pagador_cnpjcpf', 'pagador_inscricao_hash']

# Função para converter em um DataFrame

def converter_tipos(df):

    # Converte colunas numéricas para tipo numérico, formatando '\N' como NaN

    for col in colunas_numericas:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col].replace('\\N', np.nan), errors='coerce')

    # Converte colunas categóricas para category

    for col in colunas_categoricas:
        if col in df.columns:
            df[col] = df[col].astype('category')

    # Converte colunas de data para tipo datetime, formatando '\N' como NaN

    for col in colunas_data:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col].replace('\\N', np.nan), errors='coerce')

# Aplica a função de conversão de tipos para cada DataFrame

for df in [df_gf, df_gl, df_gm, df_gp, df_gt, df_gv, df_gw]:
    converter_tipos(df)

## 1.5 Tratamento de Valores Ausentes

O código abaixo passa por todos os dataframes e remove valores ausentes das colunas 'vl_pagto', 'multa' e 'qtd_acessos_pagador' e substitui esses valores por 0.

In [ ]:
# Colunas para remoção considerando que tem muitos valores ausentes

# colunas_para_remover = ['tipo_multa', 'multa']

for df in [df_gf, df_gl, df_gm, df_gp, df_gt, df_gv, df_gw]:

    # df.drop(columns=[c for c in colunas_para_remover if c in df.columns], inplace=True)

    # Substitui valores ausentes em colunas numéricas por 0

    colunas_ausentes = ['vl_pagto', 'multa', 'qtd_acessos_pagador']

    for coluna in colunas_ausentes:
        if coluna in df.columns:
            df[coluna] = df[coluna].fillna(0)

    # Trata valores ausentes em pagador_cidade

    if 'pagador_cidade' in df.columns:
        df['pagador_cidade'] = df['pagador_cidade'].cat.add_categories('Desconhecido')
        df['pagador_cidade'] = df['pagador_cidade'].fillna('Desconhecido')


## 1.6 Identificação e Tratamento de Outliers


Esse código percorre várias tabelas e, nas colunas numéricas, identifica outliers pelo intervalo interquartil (IQR) e os substitui pelos limites mínimo e máximo aceitáveis, ajustando valores extremos sem excluir dados.

In [ ]:
# Verificação em quartis

for df in [df_gf, df_gl, df_gm, df_gp, df_gt, df_gv, df_gw]:
    for col in colunas_numericas:
        if col in df.columns:

            # Quartis
            Q1 = df[col].quantile(0.25)
            Q3 = df[col].quantile(0.75)
            IQR = Q3 - Q1

            # Limites
            lim_inf = Q1 - 1.5 * IQR
            lim_sup = Q3 + 1.5 * IQR

            # Mediana da coluna
            mediana = df[col].median()

            # Substituir outliers pela mediana
            df[col] = df[col].apply(
                lambda x: mediana if (x < lim_inf or x > lim_sup) else x
            )


In [ ]:
print(df_gf.columns.tolist())


## 1.7 Pré-processamento (Normalização e Codificação)

O código cria uma função que tem a finalidade de pré-processar os dados, convertendo colunas numéricas para tipos adequados, normalizando valores quando necessário. Além disso, transforma variáveis categóricas como 'pagador_cidade', 'status_boleto', 'tipo_juros', 'tipo_multa', 'pagador_cep' e 'banco' em números com LabelEncoder, que associa cada categoria a um número correspondente, assim preparando o DataFrame para uso em modelos de Machine Learning.

In [ ]:
# Pré-processamento (One-Hot e Normalização no escopo de 0 a 10000)
colunas_numericas = ['id_grupo','id_pagador','id_beneficiario','Numero_do_boleto',
                     'vl_boleto','vl_pagto','juros','multa','valor_abatimento','qtd_acessos_pagador']
colunas_categoricas = ['status_boleto','banco','pagador_cidade','tipo_juros','tipo_multa']
colunas_data = ['data_inclusao','data_vencto','dt_pagto','pagador_dt_ultimo_acesso']
colunas_texto = ['pagador_cep','pagador_cnpjcpf','pagador_inscricao_hash']

colunas_nao_normalizar = ['id_grupo','id_beneficiario','Numero_do_boleto','id_pagador',
                          'pagador_cep','data_inclusao','data_vencto','dt_pagto','pagador_dt_ultimo_acesso']

# Criando um codificador para cada variável categórica



def preprocessar(df):
    df_proc = df

    # Normalização numérica

    num_candidatas = [c for c in colunas_numericas if c in df_proc.columns]
    num_para_normalizar = [c for c in num_candidatas if c not in colunas_nao_normalizar]

    if num_para_normalizar:
        X_num = df_proc[num_para_normalizar].copy()

        # Substitui '\N' por NaN

        X_num = X_num.replace('\\N', np.nan)

        # Converte para numérico

        X_num = X_num.apply(pd.to_numeric, errors='coerce')

        scaler = MinMaxScaler(feature_range=(0, 10000))
        arr_norm = scaler.fit_transform(X_num)
        df_norm = pd.DataFrame(arr_norm, columns=[f'{c}_norm' for c in num_para_normalizar], index=df_proc.index)
        for c in df_norm.columns:
            df_norm[c] = df_norm[c].astype('float32')
        df_proc = pd.concat([df_proc, df_norm], axis=1)
    # Conversão float64 pra float32

    for c in df_proc.select_dtypes(include='float64').columns:
        df_proc[c] = df_proc[c].astype('float32')

    # Usando LabelEncoder para variáveis categóricas


    return df_proc

O código abaixo aplica a função de pré-processamento a cada DataFrame.

In [ ]:

df_gf_pre = preprocessar(df_gf)
df_gl_pre = preprocessar(df_gl)
df_gm_pre = preprocessar(df_gm)
df_gp_pre = preprocessar(df_gp)
df_gt_pre = preprocessar(df_gt)
df_gv_pre = preprocessar(df_gv)
df_gw_pre = preprocessar(df_gw)


O código exibe o nome, as 3 primeiras linhas e o resumo de informações de cada DataFrame pré-processado.

In [ ]:
# Iteração para visualizar resultados

for nome, df_proc in {"GF": df_gf_pre, "GL": df_gl_pre, "GM": df_gm_pre, "GP": df_gp_pre, "GT": df_gt_pre, "GV": df_gv_pre, "GW": df_gw_pre}.items():
    print(f"\n {nome}")
    display(df_proc.head(3))
    print(df_proc.info())

 ## 2 Estatística Descritiva



O código abaixo imprime o nome de cada DataFrame e mostra suas estatísticas descritivas (média, desvio, mínimo, máximo, quartis) usando .describe().

In [ ]:
# Método describe aplicado

for nome, df_proc in {"GF": df_gf_pre, "GL": df_gl_pre, "GM": df_gm_pre, "GP": df_gp_pre, "GT": df_gt_pre, "GV": df_gv_pre, "GW": df_gw_pre}.items():
    print(f"\n Estatísticas {nome}")
    display(df_proc.describe())

## 3 Gráficos e Hipóteses


### 3.1 Criação de Variáveis

In [ ]:
TabelaGeral = pd.concat([df_gf_pre, df_gl_pre, df_gm_pre, df_gp_pre, df_gt_pre, df_gv_pre, df_gw_pre], ignore_index=True)
print(TabelaGeral.columns.tolist())


# Variável universal para tabela atual
TabelaAtual = df_gf_pre


Banco_encoder = LabelEncoder()
Cidade_Pagador_encoder = LabelEncoder()
Status_boleto_encoder = LabelEncoder()
Juros_encoder = LabelEncoder()
cep_encoder = LabelEncoder()
Multa_encoder = LabelEncoder()
Tipo_pagador_encoder = LabelEncoder()

Banco_encoder.fit(TabelaAtual['banco'])
TabelaAtual['banco'] = Banco_encoder.transform(TabelaAtual['banco'])

Tipo_pagador_encoder.fit(df_proc['pagador_cnpjcpf'])
df_proc['pagador_cnpjcpf'] = Tipo_pagador_encoder.transform(df_proc['pagador_cnpjcpf'])

Cidade_Pagador_encoder.fit(TabelaAtual['pagador_cidade'])
TabelaAtual['pagador_cidade'] = Cidade_Pagador_encoder.transform(TabelaAtual['pagador_cidade'])

Status_boleto_encoder.fit(TabelaAtual['status_boleto'])
TabelaAtual['status_boleto'] = Status_boleto_encoder.transform(TabelaAtual['status_boleto'])

Juros_encoder.fit(TabelaAtual['tipo_juros'])
TabelaAtual['tipo_juros'] = Juros_encoder.transform(TabelaAtual['tipo_juros'])

Multa_encoder.fit(TabelaAtual['tipo_multa'])
TabelaAtual['tipo_multa'] = Multa_encoder.transform(TabelaAtual['tipo_multa'])

cep_encoder.fit(TabelaAtual['pagador_cep'])
TabelaAtual['pagador_cep'] = cep_encoder.transform(TabelaAtual['pagador_cep'])


TabelaAtual['pagador_cidade'] = Cidade_Pagador_encoder.inverse_transform(TabelaAtual['pagador_cidade'])
TabelaAtual['status_boleto'] = Status_boleto_encoder.inverse_transform(TabelaAtual['status_boleto'])

# Convertendo Datas em string para DATETIME

TabelaAtual['data_vencto'] = pd.to_datetime(TabelaAtual['data_vencto'], errors='coerce', dayfirst=True)
TabelaAtual['data_inclusao'] = pd.to_datetime(TabelaAtual['data_inclusao'], errors='coerce', dayfirst=True)
TabelaAtual['dt_pagto'] = pd.to_datetime(TabelaAtual['dt_pagto'], errors='coerce', dayfirst=True)

# Convertendo de INT para numérico

TabelaAtual['vl_boleto'] = pd.to_numeric(TabelaAtual['vl_boleto'], errors='coerce')
TabelaAtual['juros'] = pd.to_numeric(TabelaAtual['juros'], errors='coerce')

# Criando coluna em númérico do status do boleto
print(TabelaAtual['status_boleto'])
mapping = {'REGISTRADO': 0, 'BAIXADO': 1, 'LIQUIDADO': 2}
TabelaAtual['status_num'] = TabelaAtual['status_boleto'].map(mapping)



# Dicionário de mapeamento mês
mapping = {
    1: 'Janeiro', 2: 'Fevereiro', 3: 'Março', 4: 'Abril',
    5: 'Maio', 6: 'Junho', 7: 'Julho', 8: 'Agosto',
    9: 'Setembro', 10: 'Outubro', 11: 'Novembro', 12: 'Dezembro'
}


# Criar colunas de mês
TabelaAtual['mes_pag'] = TabelaAtual['dt_pagto'].dt.month.map(mapping)
TabelaAtual['mes_inclusao'] = TabelaAtual['data_inclusao'].dt.month.map(mapping)
TabelaAtual['mes_vencimento'] = TabelaAtual['data_vencto'].dt.month.map(mapping)
ordem_meses = list(mapping.values())

TabelaAtual['dia_vencimento'] = TabelaAtual['data_vencto'].dt.day
print(TabelaAtual['dia_vencimento'])
ordem_meses = list(mapping.values())
# Filtrando boletos pagos
boletosPagos = TabelaAtual[(TabelaAtual['status_num'] == 2)]

DiasParaPagar = (boletosPagos['dt_pagto'] - boletosPagos['data_inclusao']).dt.days
TabelaAtual['dias_para_pagar'] = DiasParaPagar
DiasParaVencer = (TabelaAtual['data_vencto'] - TabelaAtual['data_inclusao']).dt.days
TabelaAtual['dias_para_vencer'] = DiasParaVencer
DiasParaPagar = DiasParaPagar.value_counts().sort_index()


dataVen = TabelaAtual['data_vencto']
dataInc = TabelaAtual['data_inclusao']
Datas = (dataVen - dataInc).dt.days

TipoData = Datas.value_counts()
TipoData = TipoData.sort_index()

mapping = {0: 'Não Pago', 1: 'Não Pago', 2: 'Pago'}
TabelaAtual['pago/n'] = TabelaAtual['status_num'].map(mapping)

### 3.2 Gráficos

Os gráficos apresentados abaixo foram utilizados como teste e aprendizado pela equipe, com o objetivo de desenvolver habilidades na criação de visualizações que, posteriormente, seriam utilizadas na criação de gráficos para as hipóteses específicas.

In [ ]:


# Gráfico de barras
plt.bar(TipoData.index, TipoData.values, edgecolor="black")
plt.title("Diferença em Dias entre Vencimento e Inclusão")
plt.xlabel("Dias de diferença")
plt.ylabel("Frequência")
plt.show()


In [ ]:
boletosPagos = TabelaAtual[TabelaAtual['status_num'] == 2]

boletosPagos['vl_boleto'] = pd.to_numeric(boletosPagos['vl_boleto'], errors='coerce')

# Remover linhas que ficaram NaN (eram inválidas)
boletosPagos = boletosPagos.dropna(subset=['vl_boleto'])

# Cria as faixas
bins = 20
boletosPagos['faixa_boleto'] = pd.cut(boletosPagos['vl_boleto'], bins=bins)

# Conta por faixa
vBoletoPagosGrouped = boletosPagos['faixa_boleto'].value_counts().sort_index()

# Gráfico
fig, ax = plt.subplots(figsize=(12,6))
ax.bar(vBoletoPagosGrouped.index.astype(str), vBoletoPagosGrouped.values, edgecolor="black")

ax.set_title("Valores dos boletos pagos")
ax.set_xlabel("Faixa de Valor do Boleto")
ax.set_ylabel("Número de Boletos Pagos")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
vBoleto = TabelaAtual['vl_boleto'].value_counts().sort_index()

plt.hist(TabelaAtual['vl_boleto'], bins=30, edgecolor="black")

plt.title("Valores dos boletos")
plt.xlabel("Valor")
plt.ylabel("Frequência")
plt.show()

In [ ]:


numeroPorCidade = TabelaAtual['pagador_cidade'].value_counts().sort_index()

plt.figure(figsize=(16,9))
sns.barplot(x="pagador_cidade", y="vl_boleto", data=TabelaAtual, estimator="mean")
plt.xticks(rotation=90)
plt.title("Média de valor por região")
plt.xlabel("Cidade")
plt.ylabel("Média de valor")
plt.show()

In [ ]:
TabelaAtual['mes_pag'] = pd.Categorical(TabelaAtual['mes_pag'], categories=ordem_meses, ordered=True)
TabelaAtual['mes_inclusao'] = pd.Categorical(TabelaAtual['mes_inclusao'], categories=ordem_meses, ordered=True)
TabelaAtual['mes_vencimento'] = pd.Categorical(TabelaAtual['mes_vencimento'], categories=ordem_meses, ordered=True)
# Agrupamento
boletos_incluidos = TabelaAtual.groupby('mes_inclusao').size()
boletos_pagos = TabelaAtual[TabelaAtual['status_num'] == 2].groupby('mes_pag').size()

# Unir em dataframe
df_mes = pd.concat([boletos_incluidos, boletos_pagos], axis=1)
df_mes.columns = ['Incluídos', 'Pagos']
df_mes = df_mes.fillna(0)

# Taxa de pagamento
df_mes['Taxa'] = (df_mes['Pagos'] / df_mes['Incluídos']) * 100

# Gráfico
fig, ax1 = plt.subplots(figsize=(12, 6))

df_mes[['Incluídos', 'Pagos']].plot(kind='bar', ax=ax1, edgecolor="black")
ax1.set_ylabel("Número de boletos")
ax1.set_xlabel("Meses")
ax1.set_title("Pagamentos vs Boletos Incluídos por Mês")


# Legendas
ax1.legend(loc="upper left")


plt.show()

## 3.3 Hipóteses

 Nesta análise, foram levantadas hipóteses relacionadas ao comportamento de pagamento de boletos pelos usuários da plataforma. O objetivo é identificar padrões e fatores que influenciam o tempo de pagamento. Essas hipóteses servem como base para exploração de dados e para a criação do modelo preditivo, auxiliando na compreensão dos fatores que impactam atrasos, pontualidade nos pagamentos e permitem identificar quais variáveis impactão ou não no pagamento de boletos.

### 3.3.1 Hipótese 1 - Dias necessários para o pagamento de boletos

**Tese:** A maioria dos boletos, tanto pagos quanto não pagos, possui um intervalo entre a data de inclusão e a data de vencimento concentrado entre 20 e 30 dias.

**Resultado:** Positivo, ambos possuem um intervalo concentrado entre 20 e 30 dias.

In [ ]:


# Gráfico de barras - Boletos Pagos
plt.bar(TipoData.index, TipoData.values, edgecolor="black")
plt.title("Diferença em Dias entre Vencimento e Inclusão")
plt.xlabel("Intervalo(dias)")
plt.ylabel("Número de boletos pagos")
plt.show()

# Gráfico de barras - Boletos Não Pagos
plt.bar(TipoData.index, TipoData.values, edgecolor="black")
plt.title("Diferença em Dias entre Vencimento e Inclusão")
plt.xlabel("Intervalo(dias)")
plt.ylabel("Número de boletos não pagos")
plt.show()

### 3.3.2 Hipótese 2 -  O período do ano influencia diretamente na taxa de pagamento de boletos.

&nbsp;&nbsp;&nbsp;&nbsp; Esta hipótese busca confirmar que a taxa de pagamento de boletos varia de acordo com o mês ou período do ano, influenciada por fatores sazonais e econômicos. Em determinados meses, como dezembro (festividades de fim de ano) ou março/abril (pagamento de impostos), pode haver uma maior pressão financeira, reduzindo a taxa de quitação de boletos. Já em períodos de maior liquidez, como meses de recebimento de 13º salário ou distribuição de bônus corporativos, a taxa de pagamento tende a aumentar. Além disso, mudanças na governança ou estratégias financeiras das empresas ao longo do ano podem impactar diretamente a capacidade de honrar pagamentos.

&nbsp;&nbsp;&nbsp;&nbsp; No gráfico abaixo, análisamos mais de 30 mil boletos entre um período de 12 meses, utilizando Python e bibliotecas como Pandas, Matplotlib e Seaborn. O eixo X representa os meses do ano, enquanto o Y representa o número de boletos. A barra em azul representa os boletos incluídos, a laranja os boletos pagos e a linha vermelha representa a taxa de pagamento em porcentagem.


In [ ]:


TabelaAtual['mes_pag'] = pd.Categorical(TabelaAtual['mes_pag'], categories=ordem_meses, ordered=True)
TabelaAtual['mes_inclusao'] = pd.Categorical(TabelaAtual['mes_inclusao'], categories=ordem_meses, ordered=True)
TabelaAtual['mes_vencimento'] = pd.Categorical(TabelaAtual['mes_vencimento'], categories=ordem_meses, ordered=True)

# Agrupamento
boletos_incluidos = TabelaAtual.groupby('mes_inclusao').size()
boletos_pagos = TabelaAtual[TabelaAtual['status_num'] == 2].groupby('mes_pag').size()

# Unir em dataframe
df_mes = pd.concat([boletos_incluidos, boletos_pagos], axis=1)
df_mes.columns = ['Incluídos', 'Pagos']
df_mes = df_mes.fillna(0)

# Taxa de pagamento
df_mes['Taxa'] = (df_mes['Pagos'] / df_mes['Incluídos']) * 100

# Gráfico
fig, ax1 = plt.subplots(figsize=(12, 6))

df_mes[['Incluídos', 'Pagos']].plot(kind='bar', ax=ax1, edgecolor="black")
ax1.set_ylabel("Número de boletos")
ax1.set_xlabel("Meses")
ax1.set_title("Pagamentos vs Boletos Incluídos por Mês")

# Eixo secundário
ax2 = ax1.twinx()
ax2.plot(df_mes.index, df_mes['Taxa'], color="red", marker="o", label="Taxa de Pagamento (%)")
ax2.set_ylabel("Taxa de Pagamento (%)")

# Legendas
ax1.legend(loc="upper left")
ax2.legend(loc="upper right")

plt.show()



&nbsp;&nbsp;&nbsp;&nbsp; A análise do gráfico revela padrões importantes no comportamento de pagamento dos boletos. Em janeiro, observa-se que a quantidade de pagamentos supera a de inclusões, o que desmonstra o impacto das festividades e despesas concentradas no fim do ano. Isso ocorre porque em dezembro há um aumento expressivo na emissão de boletos, enquanto em janeiro, período de retomada das atividades, o número de novos boletos é significativamente menor.

&nbsp;&nbsp;&nbsp;&nbsp; Além disso, nota-se que em abril a taxa de pagamentos apresenta um dos menores índices do ano, o que pode estar associado à sobrecarga financeira causada pelo pagamento de impostos obrigatórios, como o Imposto de Renda e tributos anuais. Esse padrão reforça a influência de fatores econômicos e sazonais na dinâmica de pagamento.

**Resultado:** Portanto, é possível afirmar que o período do ano influncia diretamente no pagamento de boletos.

### 3.3.3 Hipótese 3 - O tempo necessário para pagamento de boletos varia de acordo com o perfil financeiro e comportamento dos clientes

&nbsp;&nbsp;&nbsp;&nbsp; Esta hipótese busca confirmar que o tempo necessário para pagamento de boletos varia de acordo com o perfil financeiro e comportamento dos clientes. Observa-se que grande parte dos boletos é quitada dentro dos primeiros 30 dias após a inclusão, indicando que muitos clientes priorizam pagamentos imediatos. Por outro lado, uma parcela menor apresenta atrasos mais longos, podendo estar relacionada a dificuldades financeiras temporárias, menor disciplina de pagamento ou atrasos na compensação bancária.

&nbsp;&nbsp;&nbsp;&nbsp; No gráfico abaixo, o eixo X representa os dias necessários para pagar, enquanto o eixo Y representa o número de boletos quitados em cada período. Cada ponto azul indica a quantidade de boletos pagos após determinado número de dias desde a inclusão.

In [ ]:

    # Gráfico de barras
plt.plot(DiasParaPagar.index, DiasParaPagar.values, marker="o")
plt.title("Dias necessários para pagar após a inclusão")
plt.xlabel("Dias para pagar")
plt.ylabel("Número de boletos")
plt.show()


&nbsp;&nbsp;&nbsp;&nbsp; A análise do gráfico revela padrões importantes no comportamento de pagamento. Observa-se um pico significativo entre 20 e 30 dias, sugerindo que a maioria dos clientes paga dentro do prazo mensal típico. Além disso, o número de boletos pagos decresce rapidamente após 40 dias, evidenciando que atrasos prolongados são menos frequentes, mas ainda presentes, indicando a necessidade de monitoramento e estratégias de cobrança para evitar inadimplência.

&nbsp;&nbsp;&nbsp;&nbsp; **Resultado:** Portanto, é possível afirmar que o tempo de pagamento de boletos segue um padrão concentrado nos primeiros 30 dias, com uma diminuição acentuada à medida que os dias avançam.

### 3.3.4 Hipótese 4 - Boletos com valores mais altos tendem a ser pagos em um maior número de dias após a emissão, enquanto boletos de menor valor costumam ser liquidados mais rapidamente.

&nbsp;&nbsp;&nbsp;&nbsp; Essa hipótese busca afirmar se valores elevados representam um maior impacto no orçamento do cliente, já que exigem maior planejamento financeiro, o que pode gerar atrasos ou maior tempo médio até o pagamento.

&nbsp;&nbsp;&nbsp;&nbsp; Esse comportamento pode estar associado tanto ao poder aquisitivo dos clientes quanto à priorização de despesas no ciclo financeiro.

&nbsp;&nbsp;&nbsp;&nbsp; No gráfico abaixo, o eixo X representa os dias necessários para pagar o boleto e o Y o valor do boleto.

In [ ]:
DiasParaPagar = TabelaAtual[TabelaAtual['dias_para_pagar'] > 0]
DiasParaPagar['vl_boleto'] = pd.to_numeric(TabelaAtual['vl_boleto'], errors='coerce')
plt.figure(figsize=(10,5))
plt.scatter(DiasParaPagar['dias_para_pagar'], DiasParaPagar['vl_boleto'], alpha=0.6, marker="o")
plt.title("Dias necessários para pagar após a inclusão")
plt.xlabel("Dias para pagar")
plt.ylabel("Valor do boleto")
plt.show()

&nbsp;&nbsp;&nbsp;&nbsp; A partir da análise do gráfico acima, é possível identificar que não há uma tendência linear clara entre essas variáveis. A maioria dos pagamentos ocorre em até 30 dias, concentrando-se em boletos de R$ 200 a R$ 400. Casos isolados de valores acima de R$ 450 mostram pagamentos tanto rápidos quanto atrasados, sugerindo que o valor sozinho não determina o tempo de quitação. Após 40 dias, os pagamentos se tornam mais raros. Assim, o prazo para pagamento parece depender mais de fatores como comportamento do cliente, sazonalidade ou disciplina financeira, indicando que o valor do boleto deve ser analisado junto a outras variáveis em modelos preditivos.

**Resultado:** Portanto, é possível afirmar que NÃO há uma relação clara entre o valor do boleto e tempo necessário para pagar.


### 3.3.5 Hipótese 5 - A média de acesso de pessoas que pagaram o boleto é maior que as que não pagaram.

&nbsp;&nbsp;&nbsp;&nbsp; Essa hipótese busca confirmar que usuários que efetivam o pagamento de seus boletos tendem a apresentar uma quantidade de acessos à plataforma significativamente maior do que aqueles que não realizam o pagamento.

&nbsp;&nbsp;&nbsp;&nbsp; O número de acessos pode refletir o nível de engajamento do usuário com a plataforma. Um usuário mais engajado costuma ter maior contato com os serviços, maior percepção de valor e, consequentemente, maior probabilidade de honrar o pagamento. Por outro lado, usuários com baixa frequência de acessos podem demonstrar desinteresse ou menor vínculo, aumentando a chance de inadimplência.

&nbsp;&nbsp;&nbsp;&nbsp; Se a hipótese for confirmada, a quantidade de acesso pode se tornar uma forte variável na implementação do modelo preditivo, pois se torna um dos indicadores de pagamento e influencia na busca de inadimplentes.

&nbsp;&nbsp;&nbsp;&nbsp; No gráfico abaixo, o eixo X possui dois valores, os boletos pagos e não pagos, enquanto o eixo Y representa a média de acessos dos usuários de cada um desses grupos.

In [ ]:
plt.figure(figsize=(6,5))
sns.barplot(x="pago/n", y="qtd_acessos_pagador", data=TabelaAtual, estimator="mean")

plt.title("Média de acessos por status de pagamento")
plt.xlabel("Status de Pagamento")
plt.ylabel("Média de Acessos")
plt.show()


**Resultado:** Portanto, através da análise do gráfico, é possível confirmar que a média de acessos das pessoas que pagaram os boletos é consideravelmente maior que das pessoas que não pagaram.

### 3.3.6. Hipótese 6 - Média de valor entre boletos pagos e não pagos

&nbsp;&nbsp;&nbsp;&nbsp; Esta hipótese busca afirmar que a média de valor é maior para os boletos com status 'Não Pago', sugerindo que valores mais altos podem influenciar negativamente a quitação dos boletos.

In [ ]:

mapping = {0: 'Não Pago', 1: 'Não Pago', 2: 'Pago'}
TabelaAtual['pago/n'] = TabelaAtual['status_num'].map(mapping)

TabelaAtual['vl_boleto'] = pd.to_numeric(TabelaAtual['vl_boleto'], errors='coerce')
DiasParaPagar['vl_boleto'] = pd.to_numeric(TabelaAtual['vl_boleto'], errors='coerce')
plt.figure(figsize=(6,5))
sns.barplot(x="pago/n", y="vl_boleto", data=TabelaAtual, estimator="mean")

plt.title("Média de valor por status de pagamento")
plt.xlabel("Status de Pagamento")
plt.ylabel("Média do valor")
plt.show()

**Resultado**: Após a análise do gráfico, pode-se concluir que a média de valor entre boletos pagos e não pagos é praticamente igual, o que anula completamente a hipótese de que a média de valor de boletos não pagos é maior.


&nbsp;&nbsp;&nbsp;&nbsp; As hipóteses levantadas permitem direcionar análises quantitativas para verificar a influência de diferentes fatores sobre o comportamento de pagamento. A avaliação de padrões relacionados a valores, frequência de acessos e histórico de pagamentos ajuda a confirmar ou rejeitar essas hipóteses, identificando as variáveis relevantes para a criação do nosso modelo preditivo destinado a Finnet.


## 4. Criação do Modelo Preditivo

&nbsp;&nbsp;&nbsp;&nbsp; Nessa seção, foram desenvolvidos os modelos preditivos necessários para atender às entregas propostas no TAPI, cada um deles com objetivos específicos:


- Modelo de tendência de pagamento: estruturado como um problema de classificação e de aprendizado supervisionado, esse modelo prevê se o pagamento ocorrerá em dia, com atraso ou se resultará em inadimplência, fornecendo uma visão antecipada do comportamento do pagador.

- Modelo da melhor janela de pagamento provável: desenvolvido para identificar o período mais adequado em que o pagador tende a realizar o pagamento, oferecendo subsídios para ações proativas de cobrança e definição de prazos estratégicos. Esse modelo será estruturado como um problema de regressão e a aprendizagem será feita de forma supervisionada.

- Modelo de impacto projetado no fluxo de recebimentos: criado para estimar como diferentes cenários de pagamento (adimplência, atraso ou inadimplência) afetam o fluxo de caixa futuro, permitindo ao grupo avaliar riscos financeiros e tomar decisões de planejamento. Esse modelo será estruturado como um problema de regressão e a aprendizagem será feita de forma supervisionada.

### 4.1 Tratamento dos dados

&nbsp;&nbsp;&nbsp;&nbsp; Durante o processo de criação de gráficos e hipóteses, alguns dos novos dados criados durante esse processo não foram previamente tratados e normalizados, dessa forma, realizamos novamente a substituição dos valores nulos, codificação das variáveis categóricas e substituição dos valores ausentes.

### 4.1.1 Substituindo valores nulos

&nbsp;&nbsp;&nbsp;&nbsp; O código abaixo, substitui os valores ausentes preenchidos por NaN (Not a Number) das tabelas **dias_para_pagar**, **qtd_acessos_pagador_norm** e **pago/n** por 0, assim possibilitando a utilização dessas dados pelo modelo preditivo.

In [ ]:
TabelaAtual = TabelaGeral

TabelaAtual['banco'] = TabelaAtual['banco'].astype(str)
TabelaAtual['tipo_juros'] = TabelaAtual['tipo_juros'].astype(str)
TabelaAtual['tipo_multa'] = TabelaAtual['tipo_multa'].astype(str)

Banco_encoder = LabelEncoder()
Cidade_Pagador_encoder = LabelEncoder()
Status_boleto_encoder = LabelEncoder()
Juros_encoder = LabelEncoder()
cep_encoder = LabelEncoder()
Multa_encoder = LabelEncoder()
Tipo_pagador_encoder = LabelEncoder()

Banco_encoder.fit(TabelaAtual['banco'])
TabelaAtual['banco'] = Banco_encoder.fit_transform(TabelaAtual['banco'])

Tipo_pagador_encoder.fit(df_proc['pagador_cnpjcpf'])
df_proc['pagador_cnpjcpf'] = Tipo_pagador_encoder.fit_transform(df_proc['pagador_cnpjcpf'])

Cidade_Pagador_encoder.fit(TabelaAtual['pagador_cidade'])
TabelaAtual['pagador_cidade'] = Cidade_Pagador_encoder.transform(TabelaAtual['pagador_cidade'])

Status_boleto_encoder.fit(TabelaAtual['status_boleto'])
TabelaAtual['status_boleto'] = Status_boleto_encoder.transform(TabelaAtual['status_boleto'])

Juros_encoder.fit(TabelaAtual['tipo_juros'])
TabelaAtual['tipo_juros'] = Juros_encoder.transform(TabelaAtual['tipo_juros'])

Multa_encoder.fit(TabelaAtual['tipo_multa'])
TabelaAtual['tipo_multa'] = Multa_encoder.transform(TabelaAtual['tipo_multa'])

TabelaAtual['pagador_cep'] = TabelaAtual['pagador_cep'].fillna('0').astype(str)
cep_encoder.fit(TabelaAtual['pagador_cep'])
TabelaAtual['pagador_cep'] = cep_encoder.transform(TabelaAtual['pagador_cep'])


TabelaAtual['pagador_cidade'] = Cidade_Pagador_encoder.inverse_transform(TabelaAtual['pagador_cidade'])
TabelaAtual['status_boleto'] = Status_boleto_encoder.inverse_transform(TabelaAtual['status_boleto'])

# Convertendo Datas em string para DATETIME

TabelaAtual['data_vencto'] = pd.to_datetime(TabelaAtual['data_vencto'], errors='coerce', dayfirst=True)
TabelaAtual['data_inclusao'] = pd.to_datetime(TabelaAtual['data_inclusao'], errors='coerce', dayfirst=True)
TabelaAtual['dt_pagto'] = pd.to_datetime(TabelaAtual['dt_pagto'], errors='coerce', dayfirst=True)

# Convertendo de INT para numérico

TabelaAtual['vl_boleto'] = pd.to_numeric(TabelaAtual['vl_boleto'], errors='coerce')
TabelaAtual['juros'] = pd.to_numeric(TabelaAtual['juros'], errors='coerce')

# Criando coluna em númérico do status do boleto
print(TabelaAtual['status_boleto'])
mapping = {'REGISTRADO': 0, 'BAIXADO': 1, 'LIQUIDADO': 2}
TabelaAtual['status_num'] = TabelaAtual['status_boleto'].map(mapping)



# Dicionário de mapeamento mês
mapping = {
    1: 'Janeiro', 2: 'Fevereiro', 3: 'Março', 4: 'Abril',
    5: 'Maio', 6: 'Junho', 7: 'Julho', 8: 'Agosto',
    9: 'Setembro', 10: 'Outubro', 11: 'Novembro', 12: 'Dezembro'
}


# Criar colunas de mês
TabelaAtual['mes_pag'] = TabelaAtual['dt_pagto'].dt.month.map(mapping)
TabelaAtual['mes_inclusao'] = TabelaAtual['data_inclusao'].dt.month.map(mapping)
TabelaAtual['mes_vencimento'] = TabelaAtual['data_vencto'].dt.month.map(mapping)
ordem_meses = list(mapping.values())

TabelaAtual['dia_vencimento'] = TabelaAtual['data_vencto'].dt.day
print(TabelaAtual['dia_vencimento'])
ordem_meses = list(mapping.values())
# Filtrando boletos pagos
boletosPagos = TabelaAtual[(TabelaAtual['status_num'] == 2)]

DiasParaPagar = (boletosPagos['dt_pagto'] - boletosPagos['data_inclusao']).dt.days
TabelaAtual['dias_para_pagar'] = DiasParaPagar
DiasParaVencer = (TabelaAtual['data_vencto'] - TabelaAtual['data_inclusao']).dt.days
TabelaAtual['dias_para_vencer'] = DiasParaVencer
DiasParaPagar = DiasParaPagar.value_counts().sort_index()


dataVen = TabelaAtual['data_vencto']
dataInc = TabelaAtual['data_inclusao']
Datas = (dataVen - dataInc).dt.days

TipoData = Datas.value_counts()
TipoData = TipoData.sort_index()

mapping = {0: 'Não Pago', 1: 'Não Pago', 2: 'Pago'}
TabelaAtual['pago/n'] = TabelaAtual['status_num'].map(mapping)
TabelaAtual['dias_para_pagar'] = TabelaAtual['dias_para_pagar'].replace(np.nan, 0)
TabelaAtual['qtd_acessos_pagador_norm'] = TabelaAtual['qtd_acessos_pagador_norm'].replace(np.nan, 0)
TabelaAtual['qtd_acessos_pagador'] = TabelaAtual['qtd_acessos_pagador'].replace(np.nan, 0)
TabelaAtual['pago/n'] = TabelaAtual['pago/n'].fillna(0)
TabelaAtual['pago/n'] = TabelaAtual['pago/n'].replace('\\N', 0)

### 4.1.2 Transformando novas features em Label Encoder

&nbsp;&nbsp;&nbsp;&nbsp; Após a criação de novas features durante o processo de criação dos gráficos, foi necessário transformar as novas features desenvolvidas nos gráficos, hipóteses e na preparação do modelo preditivo, para o formato Label Encoder, tramsformando assim variáveis categóricas como (**mes_inclusão**, **mes_pag**, **pagador_inscricao_hash**, **pago/n (pago ou não)** e **mes_vencto**) em números que representam cada uma das categorias, para que assim, os dados possam ser entendidos pelo modelo.

In [ ]:

mesEncoder = LabelEncoder()
mesEncoder.fit(TabelaAtual['mes_inclusao'])
TabelaAtual['mes_inclusao'] = mesEncoder.fit_transform(TabelaAtual['mes_inclusao'])

mesEncoder.fit(TabelaAtual['mes_pag'])
TabelaAtual['mes_pag'] = mesEncoder.fit_transform(TabelaAtual['mes_pag'])

pagadorHash = LabelEncoder()
pagadorHash.fit(TabelaAtual['pagador_inscricao_hash'])
TabelaAtual['pagador_inscricao_hash'] = pagadorHash.fit_transform(TabelaAtual['pagador_inscricao_hash'])

TabelaAtual['pago/n'] = TabelaAtual['pago/n'].astype(str)
pagoEncoder = LabelEncoder()
pagoEncoder.fit(TabelaAtual['pago/n'])
TabelaAtual['pago/n'] = pagoEncoder.transform(TabelaAtual['pago/n'])

mesVencto = LabelEncoder()
mesVencto.fit(TabelaAtual['mes_vencimento'])
TabelaAtual['mes_vencimento'] = mesVencto.fit_transform(TabelaAtual['mes_vencimento'])



### 4.1.3 Normalizando os dados

&nbsp;&nbsp;&nbsp;&nbsp; Durante a criação do modelo preditivo, notamos que a alta dispersão de valores dos dados interfere diretamente na acurácia e resultados do modelo. Portanto, se fez necessário criar uma escala padrão para as colunas, assim transformando as linhas de algumas colunas em números em uma escala de (0,1) para melhor aproximação dos dados e assim garantindo melhores resultados e acurácia nos modelos criados.

In [ ]:
# Colunas categóricas que ainda são strings
categoricas = ['pagador_cnpjcpf', 'pago/n', 'banco', 'tipo_juros', 'tipo_multa']

for col in categoricas:
    # Preencher nulos e converter para string
    TabelaAtual[col] = TabelaAtual[col].fillna('NA').astype(str)
    # Label encoding
    le = LabelEncoder()
    TabelaAtual[col] = le.fit_transform(TabelaAtual[col])

# Agora MinMaxScaler só nas colunas numéricas
numericas = ['pagador_inscricao_hash', 'mes_pag', 'mes_inclusao', 'dias_para_pagar',
             'dias_para_vencer', 'pago/n', 'banco', 'dia_vencimento', 'mes_vencimento', 'pagador_cnpjcpf']

scaler = MinMaxScaler(feature_range=(0, 1))
for col in numericas:
    TabelaAtual[col + '_norm'] = scaler.fit_transform(TabelaAtual[[col]])


## 4.2 Modelo Preditivo de Tendência de Pagamentos

&nbsp;&nbsp;&nbsp;&nbsp; O modelo preditivo de tendência de pagamentos tem como objetivo identificar a probabilidade de o devedor realizar o próximo pagamento em dia, com atraso ou em inadimplência. Trata-se de um problema de aprendizado supervisionado, classificado como classificação multiclasse. Para viabilizar o treinamento, foi criada a variável alvo denominada “Status de Pagamento”, utilizada como variável Y no processo de modelagem preditiva.

### 4.2.1 Features

&nbsp;&nbsp;&nbsp;&nbsp; As features correspondem às variáveis explicativas utilizadas pelo modelo preditivo, ou seja, são os atributos extraídos das colunas dos boletos que fornecem informações relevantes para estimar a tendência de pagamento do usuário. Cada feature representa um aspecto do comportamento do pagador ou das características dos boletos.

#### 4.2.1.1 Variável Alvo Y (Status do Pagamento)

&nbsp;&nbsp;&nbsp;&nbsp; No desenvolvimento do modelo de tendência de pagamento, fez se necessário estabelecer, para cada boleto, qual foi o resultado do pagamento: em dia, em atraso ou inadimplente. Essa informação é representada pela variável alvo (y), que corresponde ao valor que o modelo deve aprender a prever.

&nbsp;&nbsp;&nbsp;&nbsp; Em outras palavras, enquanto as features (variáveis independentes) fornecem as características descritivas dos boletos e dos pagadores, a variável alvo indica o comportamento final que se deseja estimar. No caso em questão, a variável alvo foi definida como “Status de Pagamento”, classificando cada boleto em uma das três categorias mencionadas. Essa categorização serve de referência durante o treinamento, permitindo que o algoritmo aprenda padrões e consiga, posteriormente, prever a tendência de pagamento de novos boletos.

&nbsp;&nbsp;&nbsp;&nbsp; No código abaixo foi utilizada a data de vencimento e de pagamento para calcular o número de dias que cada boleto levou para ser pago. Se o número de dias for qualquer número positivo maior que 0, significa que o boleto foi pago em dia, se for entre -1 e -30, significa que o boleto foi pago com atraso e se tiver um valor menor que -30 (Ou seja, demorou mais de 30 dias para pagar) o boleto é considerado como inadimplente.

In [ ]:
diasNecessáriosParaPagar = (TabelaAtual['data_vencto'] - TabelaAtual['dt_pagto']).dt.days

TabelaAtual['status_do_pagamento'] = None
# Definir status do pagamento
TabelaAtual.loc[diasNecessáriosParaPagar >= 0, 'status_do_pagamento'] = 'em dia'
TabelaAtual.loc[(diasNecessáriosParaPagar <= -1) & (diasNecessáriosParaPagar >= -30), 'status_do_pagamento'] = 'atrasado'
TabelaAtual.loc[diasNecessáriosParaPagar <= -31, 'status_do_pagamento'] = 'inadimplente'
print(TabelaAtual['status_do_pagamento'].value_counts())

TabelaAtual.loc[diasNecessáriosParaPagar >= 0, 'score_boleto'] = 1
TabelaAtual.loc[(diasNecessáriosParaPagar <= -1) & (diasNecessáriosParaPagar >= -30), 'score_boleto'] =  ((diasNecessáriosParaPagar) / 31) + 1
TabelaAtual.loc[diasNecessáriosParaPagar <= -31, 'score_boleto'] = 0

# Remover coluna antiga se existir
if 'score_boleto_media' in TabelaAtual.columns:
    TabelaAtual = TabelaAtual.drop(columns=['score_boleto_media'])

# Preencher valores ausentes
TabelaAtual['status_do_pagamento'] = TabelaAtual['status_do_pagamento'].fillna('inadimplente')

# Calcular média por pagador_inscricao_hash
feature_pagador = TabelaAtual.groupby("pagador_inscricao_hash")["score_boleto"].mean()

# Criar coluna de score médio usando map (sem merge)
TabelaAtual['score_boleto_media'] = TabelaAtual['pagador_inscricao_hash'].map(feature_pagador)
# Verificar resultado
print(TabelaAtual['score_boleto_media'].value_counts())


TabelaAtual['score_boleto_media'] = TabelaAtual['score_boleto_media'].fillna(1)



#### 4.2.1.2 Variável X

&nbsp;&nbsp;&nbsp;&nbsp; O X representa o conjunto de variáveis independentes, também chamadas de features, que reúnem todas as informações utilizadas pelo modelo para aprender e prever a tendência de pagamento. No caso dos boletos, essas variáveis correspondem às colunas que descrevem características do boleto e do comportamento do pagador. Em resumo, o X é a entrada do modelo, formada por todos os atributos relevantes que permitem identificar padrões e antecipar se o pagamento ocorrerá em dia, com atraso ou inadimplente.

&nbsp;&nbsp;&nbsp;&nbsp; As features utilizadas no modelo de tendência de pagamentos foram:

- **Valor de boleto normalizado:** O valor do boleto é uma variável essencial para a predição do modelo preditivo, já que comprovamos através dos gráficos, que o valor do boleto tem impacto direto no pagamento do mesmo.
- **Quantidade de acessos:** A quantidade de acessos por pagador no boleto em específico é decisivo na predição de pagamento de boletos, já que através das hipóteses, foi comprovado que a média de acessos para os pagadores é muito maior do que aqueles que não pagam. Além disso, se a quantidade de acessos for igual a 0, há uma alta chance do boleto não ser pago, a menos que esteja em débito automático.
- **Número de inscrição do pagador em hash:** A inscrição do jogador em hash ajuda a separar dados de acordo com um pagador.
- **Juros:** O valor de juros de um boleto ajuda a identificar se o boleto será pago ou não, já que com um alto valor de juros, há uma alta chance do boleto não ser pago, enquanto um valor de juros igual a 0 representa que o boleto ainda não foi atrasado.
- **Mes_Vencimento:** Através da hipótese 2, foi comprovado que o mês de vencimento impacta diretamente no pagamento de boletos, uma vez que há meses como Janeiro, que uma alta taxa de boletos são pagos, enquanto outros como Abril (período de pagamento de impostos) que a taxa é baixíssima. Dessa forma, esse dado se torna essencial na criação de um modelo de predição de tendência de pagamento.
- **Dia de vencimento normalizado:** O dia de vencimento é uma variável importante para o modelo, pois o comportamento de pagamento pode variar de acordo com o dia do mês em que o boleto vence. Dias próximos ao início do mês, por exemplo, tendem a apresentar maior taxa de pagamento devido à entrada de salários, enquanto dias próximos ao fim do mês podem ter menor prioridade no pagamento.
- **CNPJ/CPF do pagador normalizado:** O identificador do pagador, representado em formato hash e normalizado, é essencial para distinguir diferentes perfis de clientes e capturar padrões de comportamento individualizados. Essa variável possibilita ao modelo associar o histórico de pagamentos de cada pagador à sua tendência futura, ajudando a identificar se determinados perfis estão mais propensos a pagar em dia, atrasar ou se tornar inadimplentes.

&nbsp;&nbsp;&nbsp;&nbsp; Outras features testadas no modelo preditivo, mas que não foram utilizadas na entrega:

- **Pago ou não:** a coluna **pago/n** que diz se o boleto foi pago ou não. Não foi utilizada pois causa leakage(vazamento de dados) no modelo.
- **Dias para pagar:** a coluna **dias_para_pagar** que conta o número de dias entre a data de inclusão e de pagamento. Não foi utilizada pois causa leakage(vazamento de dados) no modelo.
- **Valor de pagamento:** A coluna **vl_pagto_norm** mostra o valor de pagamento do boleto. Essa feature não foi utilizada pois além de causar leakage(vazamento de dados), não pode ser identificada antes do pagamento, portanto, para um modelo que prevê o pagamento, não há sentido em utilizá-la.


&nbsp;&nbsp;&nbsp;&nbsp; A escolha adequada das features é fundamental para a qualidade do modelo, pois são elas que fornecem os padrões que permitem ao algoritmo aprender e realizar previsões mais assertivas. Dessa forma, a etapa de definição e preparação das features garante que o modelo consiga captar as relações entre os boletos históricos e o status de pagamento futuro, servindo como base para a identificação de pagamentos em dia, com atraso ou inadimplentes.

### 4.2.2 Dividindo os dados entre X e y

&nbsp;&nbsp;&nbsp;&nbsp; O código abaixo seleciona algumas colunas do DataFrame TabelaAtual para formar as features (X) que serão usadas para treinar o modelo, e define a coluna status_do_pagamento como o alvo (y) que se quer prever. Em seguida, ele divide os dados em conjunto de treino (80%) e conjunto de teste (20%), mantendo a proporção das classes de status_do_pagamento igual em ambos os conjuntos, usando o parâmetro stratify.

In [ ]:
X = TabelaAtual[['vl_boleto_norm',
                 'qtd_acessos_pagador_norm',
                 'pagador_inscricao_hash',
                 'juros_norm',
                 'mes_vencimento_norm',
                 'dia_vencimento_norm',
                 'pagador_cnpjcpf_norm'
                 ]]

y = TabelaAtual['status_do_pagamento']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

### 4.2.3 Padronização e balanceamento dos dados

&nbsp;&nbsp;&nbsp;&nbsp; O código normaliza novamente os dados de treino e teste usando StandardScaler, centralizando em zero e ajustando a escala; depois, aplica o SMOTE no conjunto de treino para criar mais exemplos das classes minoritárias, equilibrando as classes de y_train antes de treinar o modelo.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train_scaled, y_train)

### 4.2.4 Criando o modelo preditivo

&nbsp;&nbsp;&nbsp;&nbsp; Esse código faz uma busca em grade (Grid Search) para encontrar os melhores hiperparâmetros de uma árvore de decisão. Ele testa várias combinações de profundidade máxima, tamanho mínimo de divisão, tamanho mínimo de folha e critério de divisão, usando validação cruzada de 5 folds e avaliando pelo F1 macro. Depois de encontrar a melhor árvore, treina-a nos dados de treino balanceados pelo SMOTE e faz previsões no conjunto de teste.

In [ ]:
param_grid = {
    'max_depth': [ 7, 10, None],
    'min_samples_split': [ 10, 20],
    'min_samples_leaf': [ 5, 10],
    'criterion': ['gini', 'entropy']
}

grid1 = RandomizedSearchCV(
    DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=4
)

grid1.fit(X_train_res, y_train_res)
best_tree = grid1.best_estimator_
y_pred = best_tree.predict(X_test_scaled)

### 4.2.5 Métricas

&nbsp;&nbsp;&nbsp;&nbsp; O código abaixo avalia o desempenho do modelo de classificação usando várias métricas. Primeiro, calcula métricas gerais: acurácia, que indica a proporção de previsões corretas; precisão, que mostra a proporção de acertos entre todas as previsões feitas para cada classe (em dia, atraso ou inadimplente); recall, que mede a proporção de exemplos reais de cada classe que o modelo conseguiu identificar; e F1-Score, que é a média harmônica entre precisão e recall, ponderada pelo tamanho das classes, equilibrando falsos positivos e falsos negativos.

&nbsp;&nbsp;&nbsp;&nbsp; Em seguida, o código normaliza os rótulos (y_test e y_pred) para garantir que espaços extras ou diferenças de tipo não causem erros. Depois, gera um relatório detalhado por classe usando classification_report, mostrando precisão, recall, F1-Score e suporte (quantidade de exemplos reais) para cada categoria de pagamento.

&nbsp;&nbsp;&nbsp;&nbsp; Por fim, utiliza precision_recall_fscore_support para criar um DataFrame organizado com essas métricas por classe, facilitando a visualização, análise ou exportação dos resultados. Isso permite entender tanto o desempenho geral quanto como o modelo se comporta para cada classe individualmente, identificando possíveis falhas em classes minoritárias.

In [ ]:
# --- 1) Métricas gerais
accm = accuracy_score(y_test, y_pred)
precm = precision_score(y_test, y_pred, average="weighted")
recm = recall_score(y_test, y_pred, average="weighted")
f1m = f1_score(y_test, y_pred, average="weighted")

print("Métricas do melhor modelo Decision Tree (geral):")
print(f"Acurácia: {accm:.4f}")
print(f"Precisão: {precm:.4f}")
print(f"Recall: {recm:.4f}")
print(f"F1-Score: {f1m:.4f}")


y_testm = pd.Series(y_test).astype(str).str.strip()
y_predm = pd.Series(y_pred).astype(str).str.strip()


classesm = ["inadimplente", "em dia", "atrasado"]
print(classification_report(y_test, y_pred, labels=classesm, target_names=classesm, zero_division=0))


precision, recall, f1, support = precision_recall_fscore_support(
    y_testm, y_predm, labels=classesm, zero_division=0
)

metrics_dfm = pd.DataFrame({
    "Classe": classesm,
    "Precisão": precision,
    "Recall": recall,
    "F1-Score": f1,
    "Support": support
})

# Torna as variáveis principais deste modelo acessíveis globalmente
import inspect

# Coleta todas as variáveis locais da célula atual
local_vars = dict(inspect.currentframe().f_locals)

# Filtra só as variáveis úteis (evita funções, módulos e dados muito grandes)
variaveis_interessantes = {
    k: v for k, v in local_vars.items()
    if not k.startswith("__") and not callable(v) and not isinstance(v, type(inspect))
}

# Atualiza o escopo global com as variáveis dessa célula
globals().update(variaveis_interessantes)

print(f" Variáveis exportadas para o escopo global: {list(variaveis_interessantes.keys())}")

print(metrics_dfm)

### 4.2.6 Gráficos

&nbsp;&nbsp;&nbsp;&nbsp; Esses gráficos permitem visualizar de forma clara o desempenho do modelo. O primeiro é um gráfico de barras que mostra a precisão, o recall e o F1-Score para cada classe do modelo (inadimplente, em dia e em atraso), permitindo comparar rapidamente quais classes o modelo prevê melhor ou pior.

In [ ]:
metrics_dfm.set_index("Classe", inplace=True)
metrics_dfm.plot(kind="bar", figsize=(8,5))
plt.title("Comparação de métricas por classe")
plt.ylabel("Score")
plt.ylim(0,1)
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.show()



&nbsp;&nbsp;&nbsp;&nbsp; O segundo é um heatmap da matriz de confusão, que indica os acertos e erros do modelo para cada classe, com o eixo x representando as classes previstas e o eixo y as classes reais, facilitando a identificação de quais classes o modelo confunde mais e onde ele acerta ou erra.

In [ ]:

cm = confusion_matrix(y_testm, y_predm)
labels = ["Inadimplente", "Em dia", "Em atraso"]

plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
plt.xlabel("Predito")
plt.ylabel("Real")
plt.title("Matriz de Confusão - Decision Tree")
plt.show()



&nbsp;&nbsp;&nbsp;&nbsp; O modelo consegue prever tendências gerais de pagamento, mas tem um baixo desempenho entre as classes, acertando mais as majoritárias (“em dia”) e confundindo as minoritárias (“inadimplente” e “atrasado”). Após remover variáveis com data leakage como **vl_pagto**, **vl_ratio** e **dias_para_pagar**, a acurácia caiu, mas o modelo passou a generalizar melhor. Gráficos e matriz de confusão mostram claramente onde ele acerta e erra, indicando que há espaço para melhorar com features históricas ou modelos mais robustos.

&nbsp;&nbsp;&nbsp;&nbsp; Como próximos passos para este modelo preditivo, definimos melhorar a acurácia do modelo, para garantir uma maior taxa de acertos nas previsões e usar os dados de todas as tabelas disponibilizadas pela empresa parceira, assim acrescentando valor ao produto oferecido e entregando melhores resultados ao projeto.

## 4.3 Modelo Preditivo B de Tendência de Pagamentos

### 4.3.1 Features


Para treinar um modelo é necessário selecionar as **variáveis independentes (X)**, que são usadas como entrada para realizar a previsão, e a **variável dependente (y)**, que representa o valor a ser previsto.  

Neste modelo, foram reutilizadas as mesmas variáveis **X** do modelo anterior:  
- valor do boleto,  
- quantidade de acessos do pagador,  
- hash da inscrição do pagador,  
- juros,  
- mês do vencimento,  
- dia do vencimento,  
- tipo do pagador (CPF ou CNPJ).
- score médio de pagamento  

Além dessas, foi adicionada uma **nova feature derivada**:  
- **score_boleto_media** → representa o score médio de cada pagador, calculado com base no histórico de pagamentos. Essa métrica foi criada para capturar padrões individuais de risco de inadimplência.  

A variável **y** é o **status do pagamento** (em dia, em atraso ou inadimplente).  

### Categorias de Features
- **Variáveis Originais (normalizadas):** informações referentes ao boleto, ao pagador e ao histórico de pagamentos.  
- **Feature Derivada:** métrica construída a partir dos dados históricos para enriquecer o modelo.  


In [ ]:
X = TabelaAtual[['vl_boleto_norm',
                 'qtd_acessos_pagador_norm',
                 'pagador_inscricao_hash',
                 'juros_norm',
                 'mes_vencimento_norm',
                 'dia_vencimento_norm',
                 'pagador_cnpjcpf_norm',
                 "score_boleto_media"
                 ]]

y = TabelaAtual['status_do_pagamento']

#### 4.3.2 Dividindo dados nos eixos x e y


- **X (features preditivas):** contém as variáveis explicativas normalizadas e criadas, incluindo **`score_boleto_media`**.  
- **y (variável alvo):** representa a classe de status do pagamento, podendo assumir três valores:  
  - Inadimplente  
  - Em dia  
  - Atrasado  


In [ ]:


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

### 4.3.3 Padronização e balanceamento refeita

#### Padronização
Os dados foram repadronizados para garantir que todas as **features** estivessem na mesma escala, evitando que variáveis com magnitudes maiores dominassem o treinamento do modelo.

#### Balanceamento
Como a distribuição das classes apresentava certo desbalanceamento, foram aplicadas técnicas de **balanceamento de classes**.  
Esse processo assegura que o modelo não fique enviesado em prever majoritariamente a classe mais frequente.  

Em muitos cenários de treinamento, as classes da variável **y** não estão igualmente representadas, gerando um **desbalanceamento**. Isso pode levar o modelo a ter uma "falsa precisão".  
Por exemplo: ao escolher uma letra aleatória do alfabeto, a chance de ser uma vogal é naturalmente menor que a de ser uma consoante. Se um modelo previsse sempre "consoante", ele pareceria estar correto na maioria dos casos, mas estaria enviesado.  

Para corrigir esse problema, foi utilizada a técnica **SMOTE (Synthetic Minority Over-sampling Technique)**, que gera amostras sintéticas das classes minoritárias, equilibrando os dados e reduzindo previsões "corretas" baseadas apenas na sorte.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train_scaled, y_train)

### 4.3.4 Criando o Modelo B  

O **Modelo B** foi definido como o modelo preditivo geral do projeto.  
Para sua implementação, foi utilizado o algoritmo **Random Forest Classifier**, que se baseia em múltiplas árvores de decisão combinadas em *ensemble*.  

A escolha do Random Forest se deu pelo fato de o **Modelo A** ter utilizado apenas uma árvore de decisão. Ao aumentar o número de árvores, a robustez do modelo cresce, reduzindo a chance de **overfitting** e proporcionando previsões mais consistentes.  

### Configuração  
- **Algoritmo utilizado:** Random Forest  
  - O algoritmo combina diversas árvores de decisão para gerar a previsão final por meio de **voto majoritário**, resultando em maior estabilidade e precisão.  

### Hiperparâmetros  
Hiperparâmetros são configurações que controlam como o modelo aprende e podem influenciar diretamente o desempenho, evitando *overfitting*, balanceando respostas e controlando a complexidade do modelo.  
Os principais hiperparâmetros utilizados foram:  
- **max_depth:** profundidade máxima da árvore, isto é, quantos níveis de divisão podem ser feitos.  
- **min_samples_split:** número mínimo de amostras necessárias para dividir um nó.  
- **min_samples_leaf:** número mínimo de amostras em um nó folha (nó terminal).  
- **criterion:** função usada para medir a qualidade da divisão (ex.: `gini` ou `entropy`).  
- **random_state:** fixado para garantir reprodutibilidade dos experimentos.  

### Vantagens do Random Forest  
- Redução de **overfitting** em comparação a uma árvore de decisão única.  
- Melhor **generalização** para dados não vistos.  
- **Robustez** na classificação de casos complexos, devido ao voto majoritário entre várias árvores.  


In [ ]:
param_grid = {
    'max_depth': [7],
    'min_samples_split': [ 10, 30],
    'min_samples_leaf': [ 5, 20],
    'criterion': ['gini']
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42, class_weight='balanced', n_jobs=-1),
    param_grid,
    cv=3,
    n_jobs=-1,
)

grid.fit(X_train_scaled, y_train)
best_tree = grid.best_estimator_
y_predm = best_tree.predict(X_test_scaled)

### 4.3.5 Métricas

Para mensurar o desempenho do modelo, foram utilizadas métricas amplamente aplicadas em problemas de classificação.  
Essas métricas permitem analisar tanto a performance global do modelo quanto o desempenho específico em cada classe.

- **Acurácia (Accuracy):** proporção de previsões corretas em relação ao total de observações.  
- **Precisão (Precision):** avalia, dentre as previsões positivas de uma classe, quantas foram realmente corretas.  
- **Recall (Sensibilidade):** mede a capacidade do modelo de identificar corretamente os exemplos positivos de uma classe.  
- **F1-Score:** média harmônica entre Precisão e Recall, balanceando as duas métricas.

Essas métricas foram calculadas considerando todas as classes com o parâmetro `average="weighted"`, o que pondera os resultados pelo número de ocorrências de cada classe.

#### 4.3.5.1 Relatório por Classe

Além das métricas gerais, foi gerado um **classification report** e uma tabela detalhada para cada classe:  
- **Classes avaliadas:**  
  - Inadimplente  
  - Em dia  
  - Atrasado  

Para cada classe, são reportados:  
- **Precisão (Precision):** qualidade das previsões positivas da classe.  
- **Recall:** capacidade de recuperação da classe pelo modelo.  
- **F1-Score:** equilíbrio entre precisão e recall.  
- **Support:** número de exemplos reais da classe no conjunto de teste.

Esse detalhamento possibilita identificar em quais classes o modelo apresenta maior ou menor desempenho, auxiliando na análise crítica de sua robustez.


In [ ]:
# --- 1) Métricas gerais
accm2 = accuracy_score(y_testm2, y_predm2)
precm2 = precision_score(y_testm2, y_predm2, average="weighted")
recm2 = recall_score(y_testm2, y_predm2, average="weighted")
f1m2 = f1_score(y_testm2, y_predm2, average="weighted")

print("Métricas do melhor modelo Decision Tree (geral):")
print(f"Acurácia: {accm2:.4f}")
print(f"Precisão: {precm2:.4f}")
print(f"Recall: {recm2:.4f}")
print(f"F1-Score: {f1m2:.4f}")


y_testm2 = pd.Series(y_testm2).astype(str).str.strip()
y_predm2 = pd.Series(y_predm2).astype(str).str.strip()


classesm2 = ["inadimplente", "em dia", "atrasado"]
print(classification_report(y_testm2, y_predm2, labels=classesm2, target_names=classesm2, zero_division=0))


precision, recall, f1, support = precision_recall_fscore_support(
    y_testm2, y_predm2, labels=classesm2, zero_division=0
)

metrics_dfm2 = pd.DataFrame({
    "Classe": classesm2,
    "Precisão": precision,
    "Recall": recall,
    "F1-Score": f1,
    "Support": support
})

# Torna as variáveis principais deste modelo acessíveis globalmente
import inspect

# Coleta todas as variáveis locais da célula atual
local_vars = dict(inspect.currentframe().f_locals)

# Filtra só as variáveis úteis (evita funções, módulos e dados muito grandes)
variaveis_interessantes = {
    k: v for k, v in local_vars.items()
    if not k.startswith("__") and not callable(v) and not isinstance(v, type(inspect))
}

# Atualiza o escopo global com as variáveis dessa célula
globals().update(variaveis_interessantes)

print(f" Variáveis exportadas para o escopo global: {list(variaveis_interessantes.keys())}")

print(metrics_dfm2)

### 4.3.6 Gráficos Modelo B

**Visualização (gráficos):**  
Modelos preditivos retornam grande volume de dados, por isso é essencial utilizar gráficos para facilitar a interpretação.  

Os gráficos utilizados neste modelo foram:  
- **Gráfico de colunas:** para visualizar as métricas de cada categoria de status de pagamento.  
- **Mapa de calor da matriz de confusão:** para mostrar de forma clara como cada dado foi previsto em relação ao seu status real.


In [ ]:
metrics_dfm2.set_index("Classe", inplace=True)
metrics_dfm2.plot(kind="bar", figsize=(8,5))
plt.title("Comparação de métricas por classe")
plt.ylabel("Score")
plt.ylim(0,1)
plt.xticks(rotation=0)
plt.legend(loc="lower right")
plt.show()

In [ ]:

cm = confusion_matrix(y_testm2, y_predm2)
labels = ["Inadimplente", "Em dia", "Em atraso"]

plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
plt.xlabel("Predito")
plt.ylabel("Real")
plt.title("Matriz de Confusão - Decision Tree")
plt.show()


### 4.2.7 Explicação do Modelo Preditivo (Modelo B)

O **Modelo B** foi desenvolvido para prever o **status do pagamento** (inadimplente, em dia ou atrasado) utilizando como base o **Random Forest Classifier**, um algoritmo que combina várias árvores de decisão em um *ensemble*, aumentando a robustez e reduzindo o risco de *overfitting*.

#### Features Utilizadas
O modelo considera tanto variáveis originais quanto derivadas:

- **Variáveis originais (normalizadas):**  
  - valor do boleto  
  - quantidade de acessos do pagador  
  - hash da inscrição  
  - juros  
  - mês e dia de vencimento  
  - tipo do pagador (CPF ou CNPJ)
  - score médio de pagamento

- **Feature derivada:**  
  - **score_boleto_media** → representa o score médio de cada pagador, capturando padrões históricos de risco de inadimplência

A variável alvo **y** é o **status do pagamento**.

#### Padronização e Balanceamento
- **Padronização:** todas as features foram colocadas na mesma escala para evitar que variáveis com magnitudes maiores dominassem o modelo.  
- **Balanceamento:** como as classes estavam desbalanceadas, foi utilizada a técnica **SMOTE** para gerar amostras sintéticas das classes minoritárias, garantindo que o modelo não previsse apenas a classe majoritária por sorte.

#### Configuração e Hiperparâmetros
- **max_depth:** profundidade máxima das árvores  
- **min_samples_split:** número mínimo de amostras para dividir um nó  
- **min_samples_leaf:** número mínimo de amostras em um nó folha  
- **criterion:** função de qualidade da divisão (ex.: `gini`)  
- **random_state:** fixado para reprodutibilidade

**Vantagens do Random Forest:** maior generalização, redução de overfitting e robustez na classificação de casos complexos.

#### Visualização e Interpretação
Para interpretar o modelo, foram utilizados gráficos:

- **Gráfico de colunas:** visualiza métricas por categoria de status  
- **Mapa de calor da matriz de confusão:** mostra previsões versus valores reais  

Além disso, o **SHAP (SHapley Additive exPlanations)** foi utilizado para explicar como cada feature contribuiu para as previsões do modelo:

- Permite analisar **previsões individuais**, mostrando se cada variável aumentou ou diminuiu a probabilidade de cada classe  
- Gera **visualizações globais**, mostrando quais features tiveram maior impacto no modelo  
- Complementa os gráficos tradicionais, trazendo transparência ao funcionamento do modelo e permitindo entender o **porquê das decisões** do Random Forest


In [ ]:
import shap
# Crie o explainer para o modelo RandomForest treinado acima
explainer = shap.TreeExplainer(best_tree)
# Amostra para o SHAP (pode ajustar o número se necessário)
X_sample = shap.sample(X_test, 10000, random_state=42)
# Calcula os valores SHAP
shap_values = explainer.shap_values(X_sample)
# Gráfico de resumo das importâncias das features
shap.summary_plot(shap_values, X_sample, feature_names=X.columns)
shap.summary_plot(shap_values, X_sample, feature_names=X.columns, plot_type="bar")

## 4.4 Modelo de Tendência de Pagamentos C

&nbsp;&nbsp;&nbsp;&nbsp; Este novo modelo faz uso do LightGBM (Light Gradient Boosting Machine), um algoritmo de Machine Learning desenvolvido para lidar com grandes volumes de dados de forma rápida e eficiente. O LightGBM é uma implementação otimizada do Gradient Boosting, técnica que constrói diversas árvores de decisão de maneira sequencial, em que cada nova árvore busca corrigir os erros das anteriores. Essa abordagem permite capturar relações complexas entre variáveis e, ao mesmo tempo, manter alta performance. No contexto do nosso problema, o LightGBM é utilizado para identificar a tendência de pagamento dos usuários, oferecendo previsões mais precisas e escaláveis mesmo em bases de dados muito extensas.

### 4.4.1 Features


&nbsp;&nbsp;&nbsp;&nbsp;  Nesse caso, definimos quais variáveis vão compor o conjunto de entrada (X) e qual será a variável de saída (y). Em X, estão todas anteriormente selecionadas pelo modelo B, dessa forma, já utiliza da nova feature (score_boleto_media) para identificar a tendência de pagamento do usuário. Já em y, coloquei o status do pagamento, que é o resultado que queremos que o modelo aprenda a prever. Dessa forma, o problema está lidando com uma tarefa de classificação supervisionada, em que o modelo usará os dados dos boletos e dos pagadores para estimar se um pagamento será realizado ou não.

In [ ]:
X = TabelaAtual[['vl_boleto_norm',
                 'qtd_acessos_pagador_norm',
                 'juros_norm',
                 'mes_vencimento_norm',
                 'dia_vencimento_norm',
                'pagador_cnpjcpf_norm',
                 "score_boleto_media"
                 ]]

y = TabelaAtual['status_do_pagamento']

### 4.4.2 Dividindo dados entre treino e teste

&nbsp;&nbsp;&nbsp;&nbsp; Nessa parte do código, os dados são dividos entre teste e treino, sendo 20% reservado para teste e o restante dos 80% totalmente guardado para o treinamento do modelo. O X_train se refere ao X de treino, X_test se refere ao X de treino, o Y_train se refere ao y de treino e o y_test se refere ao Y de teste.

In [ ]:


X_train, X_test, y_trainm3, y_testm3 = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

### 4.4.3 Padronização e balanceamento refeita

&nbsp;&nbsp;&nbsp;&nbsp; O trecho aplica uma padronização nos dados de treino e teste, transformando as variáveis preditoras para que tenham média zero e desvio padrão igual a um. Em seguida, utiliza a técnica SMOTE para realizar o balanceamento da base de treino, gerando novas amostras sintéticas da classe minoritária e equilibrando a distribuição entre as classes antes do treinamento do modelo.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train_scaled, y_train)

### 4.4.4 Criando o modelo

&nbsp;&nbsp;&nbsp;&nbsp; Como dito anteriormente, nesse modelo foi utilizado o LightGBM para identificar a tendência de pagamento do usuário. Para alcançar esse resultado, se fez necessário escolher os melhores hiper parâmetros possíveis para o nosso problema. Os hiper parâmetros escolhidos foram:

- device="gpu": O parâmetro **device** seleciona qual componente do computador será utilizado para renderizar o modelo. Para esse modelo, utilizamos da **GPU**, que acelera o treino, pois é consideravelmente menos utilizado que a CPU.
- objective="multiclass": O parâmetro **objective** define se o Y possui uma ou mais classes. Como nesse modelo utilizamos 3 classes, selecionamos como **"multiclasse"**.
- boosting_type="gbdt": O parâmetro **boosting_type** define a forma no qual o modelo gerará as ároves. Selecionamos o **"gbdt"** que utiliza de árvores anteriores para aprimorar o modelo.
- num_leaves=255: O **num_leaves** define o número máximo de folhas por árvore, assim controlando capacidade do modelo. Selecionamos **255** um número razoável para garantir a eficiência e velocidade do modelo.
- max_depth=-1: O **max_depth** define a profundidade do modelo. Escolhemos como **-1**, ou seja, sem limite fixo de profundidade, o modelo decide.
- learning_rate=0.05: O **learning_rate** define a taxa de aprendizado do modelo, um menor valor como o **0.05** exige mais árvores mas entrega maior estabilidade.
- n_estimators=2000: O **n_estimators** define o número total de árvores no ensemble. Escolhemos como 2000, um número razoávelmente grande, mas que se faz necessário dado o largo tamanho do dataset.
- min_child_samples=100: O **min_child_samples** é mínimo de exemplos por folha e reduz risco de overfitting. Definimos como 100.
- min_split_gain=0.0: O **min_split_gain** define o ganho mínimo exigido para dividir um nó.
- subsample=0.8 → O **subsample** define a porcentagem de amostra de cada árvore. No nosso modelo, cada árvore usa 80% das amostras em cada árvore (bagging).
- subsample_freq=1: O **subsample_freq** define a frequência com qual o subsample acontecerá. Definimos como **1**, dessa forma, ele aplica o subsample em cada iteração.
- colsample_bytree=0.8: O **colsample_bytree** define a proporção de variáveis usadas em cada árvore. No nosso caso, cada árvore usa **80%** das variáveis.
- reg_alpha=1.0: O **reg_alpha** é a regularização L1, que ajuda na seleção de variáveis, forçando alguns pesos a zero. Definimos como **1.0**.
- reg_lambda=1.0: O **reg_lambda** é a regularização L2, que penaliza pesos muito altos para reduzir overfitting. Definimos como **1.0**.
- n_jobs=-1: O **n_jobs** define o número de núcleos da CPU a serem utilizados. Definimos como **-1**, para usar todos os núcleos disponíve

&nbsp;&nbsp;&nbsp;&nbsp; Para esse modelo, não utilizamos o GridSearch, já que com um datasets tão grande, encontramos dificuldade em testar diferentes hiperparâmetros. Dessa forma, testamos valores compatíveis de a acordo com o contexto do projeto e a profundidade dos dados.

In [ ]:
# Versão Robusta do LightGBM

# model = lgb.LGBMClassifier(
#     device="cpu",
#     objective="multiclass",   # ou "binary", dependendo do seu problema
#     boosting_type="gbdt",

#     # Estrutura das árvores
#     num_leaves=255,          # folhas maiores aumentam capacidade, mas não exagere (>512 começa a pesar)
#     max_depth=-1,            # deixa o modelo decidir (ou limite para ~15 se overfitting)

#     # Velocidade e regularização
#     learning_rate=0.05,      # taxa menor + mais árvores = mais estável
#     n_estimators=2000,       # mais árvores para compensar o learning_rate baixo
#     min_child_samples=100,   # exige mais amostras por folha (evita overfitting em dataset grande)
#     min_split_gain=0.0,
#     subsample=0.8,           # bagging (reduz correlação entre árvores)
#     subsample_freq=1,
#     colsample_bytree=0.8,    # usa 80% das features em cada árvore

#     # Regularização L1/L2
#     reg_alpha=1.0,           # L1
#     reg_lambda=1.0,          # L2

#     # Performance
#     n_jobs=-1,               # usa todos os núcleos
#     random_state=42,

#     metric="multi_logloss",
#     class_weight='balanced'  # Ajuste conforme necessário
# )
# model.fit(X_train_res, y_train_res)

#Versão otimizada do LightGBM para exibição de resultados

model = lgb.LGBMClassifier(
    device="cpu",
    objective="multiclass",   # ou "binary", dependendo do seu problema
    boosting_type="gbdt",

    # Estrutura das árvores
    num_leaves=50,          # folhas maiores aumentam capacidade, mas não exagere (>512 começa a pesar)
    max_depth=-1,            # deixa o modelo decidir (ou limite para ~15 se overfitting)

    # Velocidade e regularização
    learning_rate=0.05,      # taxa menor + mais árvores = mais estável
    n_estimators=100,       # mais árvores para compensar o learning_rate baixo
    min_child_samples=20,   # exige mais amostras por folha (evita overfitting em dataset grande)
    min_split_gain=0.0,
    subsample=0.8,           # bagging (reduz correlação entre árvores)
    subsample_freq=1,
    colsample_bytree=0.8,    # usa 80% das features em cada árvore

    # Regularização L1/L2
    reg_alpha=1.0,           # L1
    reg_lambda=1.0,          # L2

    # Performance
    random_state=42,

    metric="multi_logloss",
    class_weight='balanced'  # Ajuste conforme necessário
)
model.fit(X_train_res, y_train_res)



### 4.4.5 Métricas



&nbsp;&nbsp;&nbsp;&nbsp; Esse código faz a avaliação do desempenho do modelo treinado com o LightGBM. Primeiro, ele gera as previsões (y_pred) com base nos dados de teste e calcula as principais métricas globais: acurácia, precisão, recall e F1-Score. Em seguida, ele ajusta os valores das variáveis alvo e previstas para garantir consistência e organiza a avaliação por classe.

&nbsp;&nbsp;&nbsp;&nbsp; Foram consideradas três categorias de status de pagamento: “inadimplente”, “em dia” e “atrasado”. O relatório de classificação mostra o desempenho do modelo em cada classe, permitindo identificar diferenças de acerto entre os grupos. Por fim, os resultados são organizados em um DataFrame, detalhando para cada classe a precisão (proporção de previsões corretas dentro da classe), o recall (capacidade de identificar corretamente os casos da classe), o F1-Score (harmonia entre precisão e recall) e o suporte (quantidade real de amostras da classe no conjunto de teste).


In [ ]:
y_predm3 = model.predict(X_test_scaled)

accm3 = accuracy_score(y_test, y_pred)
precm3 = precision_score(y_test, y_pred, average="weighted")
recm3 = recall_score(y_test, y_pred, average="weighted")
f1m3 = f1_score(y_test, y_pred, average="weighted")

print("Métricas do melhor modelo Decision Tree (geral):")
print(f"Acurácia: {accm3:.4f}")
print(f"Precisão: {precm3:.4f}")
print(f"Recall: {recm3:.4f}")
print(f"F1-Score: {f1m3:.4f}")


y_testm3 = pd.Series(y_test).astype(str).str.strip()
y_predm3 = pd.Series(y_pred).astype(str).str.strip()


classesm3 = ["inadimplente", "em dia", "atrasado"]
print(classification_report(y_testm3, y_predm3, labels=classesm3, target_names=classesm3, zero_division=0))


precision, recall, f1, support = precision_recall_fscore_support(
    y_testm3, y_predm3, labels=classesm3, zero_division=0
)

metrics_dfm3 = pd.DataFrame({
    "Classe": classesm3,
    "Precisão": precision,
    "Recall": recall,
    "F1-Score": f1,
    "Support": support
})

# Torna as variáveis principais deste modelo acessíveis globalmente
import inspect

# Coleta todas as variáveis locais da célula atual
local_vars = dict(inspect.currentframe().f_locals)

# Filtra só as variáveis úteis (evita funções, módulos e dados muito grandes)
variaveis_interessantes = {
    k: v for k, v in local_vars.items()
    if not k.startswith("__") and not callable(v) and not isinstance(v, type(inspect))
}

# Atualiza o escopo global com as variáveis dessa célula
globals().update(variaveis_interessantes)

print(f" Variáveis exportadas para o escopo global: {list(variaveis_interessantes.keys())}")

print(metrics_dfm3)

&nbsp;&nbsp;&nbsp;&nbsp; Os resultados obtidos mostram que o modelo tem uma acurácia geral de aproximadamente 78,8%, com melhor desempenho nas classes “inadimplente” e “atrasado”, que apresentam F1-Scores acima de 0,80. Já a classe “em dia” apresentou desempenho inferior, com F1-Score próximo de 0,64, indicando maior dificuldade do modelo em distinguir corretamente essa categoria.

### 4.4.6 Gráficos

In [ ]:

cm = confusion_matrix(y_test, y_pred)
labels = ["Inadimplente", "Em dia", "Em atraso"]

plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=labels, yticklabels=labels)
plt.xlabel("Predito")
plt.ylabel("Real")
plt.title("Matriz de Confusão - Decision Tree")
plt.show()

## 4.5 Comparação dos Modelos de Tendência de Pagamento


&nbsp;&nbsp;&nbsp;&nbsp; Com objetivo de resolver o problema proposto pela Finnet com maior precisão e chance de acerto possível, o grupo desenvolveu 3 modelos com o mesmo objetivo: Identificar a tendência de pagamento do usuário, se será em dia, com atraso ou inadimplência. Através da criação desses três modelos, se tornou possível alterar os algoritmos de classificação, features utilizadas e hiperparâmetros específicos para cada caso, assim entregando maior valor para a empresa parceira e garantindo que o modelo entregue possui qualidade técnica e eficácia na resolução do problema.

&nbsp;&nbsp;&nbsp;&nbsp; Cada modelo, utilizou de um diferente algoritmo de classificação, sendo eles:
- Decision Tree Classifier para o modelo A.
- Random Forest Classifier para o modelo B.
- Light Gradient Boost Machine para o modelo C.

&nbsp;&nbsp;&nbsp;&nbsp;A diferença entre o método de classificação dos modelos é: O Decision Tree cria uma árvore de decisão para identificar a tendência de pagamentos do usuário, O Random Forest utiliza da criação de múltiplas árvores de decisão aleatórias e faz uma média de acurácia entre as árvores, e o Light Gradient Boosting também cria várias árvores de decisão, mas utiliza das árvores anteriores para aprender e aprimorar a árvore que surgirá posteriormente.

&nbsp;&nbsp;&nbsp;&nbsp;Cada um desses modelos, possui características únicas, que se tornam pontos positivos para a escolha do modelo final, elas são:


**Decision Tree Classifier:**
- Rápido para treinar.
- Funciona bem sem normalização dos dados.
- Não lida tão bem com uma grande quantidade de dados.
- Utilizado para modelos mais "simples".

**Random Forest Classifier:**

- Reduz overfitting das árvores.
- Bom desempenho sem muito tuning.
- Pode ser mais pesado para dados enormes

**LGBM:**

- Pode-se usar GPU para processamento (Ou seja, mais rápido).
- Ótimo para problemas desbalanceados (class_weight).
- Lida melhor com uma grande quantidade de dados.

&nbsp;&nbsp;&nbsp;&nbsp; Após observar as características de cada algoritmo, a escolha parece clara: para o tamanho do dataset e o desbalanceamento das classes, o Modelo C (LightGBM) apresenta-se como a opção mais viável. Contudo, a definição do modelo preditivo não deve se basear apenas em aspectos teóricos. É necessário avaliar se os resultados práticos justificam essa escolha.

&nbsp;&nbsp;&nbsp;&nbsp; Para isso, na seção anterior, realizamos o treinamento e teste de cada um dos modelos candidatos, comparando-os pelas métricas de Acurácia, Precisão, Revocação (Recall) e F1-Score. E os resultados obtidos através deles foram:

|  Modelo  | Acurácia | Precisão |  Recall  | F1-Score |
|----------|----------|----------|----------|----------|
|     A    |    63%   |    65%   |    64%   |    64%   |
|     B    |    63%   |    67%   |    63%   |    63%   |
|     C    |    75%   |    76%   |    75%   |    76%   |

&nbsp;&nbsp;&nbsp;&nbsp; Através dos resultados obtidos, é possível identicar que na média das métricas para cada uma das classes (em dia, atraso e inadimplente), o modelo C foi o que apresentou maior acurácia, precisão, revocação e F1-Score entre os modelos analisados. Além disso, os modelos B e C utilizam de uma nova feature, chamada score_boleto_media, que utiliza do histórico de pagamentos anteriores de cada devedor para identificar sua tendência de pagamento futura, o que pode influenciar diretamente nos resultados distintos entre os modelos. A equipe também comparou o tempo que cada um dos modelos precisa para ser executado, o resultado foi:

|  Modelo  | Tempo necesário para execução |
|----------|-------------------------------|
|     A    |             6min              |
|     B    |             7min              |
|     C    |            36min              |

&nbsp;&nbsp;&nbsp;&nbsp; A diferença de tempo observada entre os modelos está diretamente relacionada aos hiperparâmetros e à estratégia de busca utilizada em cada caso. Os Modelos A e B (Árvore de Decisão e Random Forest) foram ajustados por meio do método GridSearchCV, que realiza uma busca exaustiva dentro dos limites estabelecidos para cada hiperparâmetro. Embora esse procedimento garanta uma seleção sistemática, ele aumenta consideravelmente o tempo de execução, pois o modelo é treinado diversas vezes com diferentes combinações de parâmetros.

&nbsp;&nbsp;&nbsp;&nbsp; Por outro lado, o Modelo C (LightGBM) não utilizou o GridSearch, mas de hiperparâmetros já definidos na criação do modelo. Apesar de teoricamente o Modelo C ser mais rápido na teoria, na prática, ele levou um tempo elevado para execução. Isso porque ele utiliza de alguns com valores elevados, como o n_estimators = 2000, que implica na construção de até duas mil árvores de decisão. À primeira vista, esse número poderia sugerir um custo computacional muito maior do que os modelos A e B. No entanto, o LightGBM foi projetado com técnicas de otimização que reduzem drasticamente o tempo de treinamento. Essas características permitem que o LightGBM treine milhares de árvores em menos tempo do que seria necessário para treinar algumas centenas em um Random Forest tradicional. Portanto, apesar do maior número de estimadores, o Modelo C alcançou melhor desempenho sem comprometer a eficiência, demonstrando sua adequação para cenários com grande volume de dados e classes desbalanceadas.

&nbsp;&nbsp;&nbsp;&nbsp; Além dos pontos já mencionados, é importante destacar que o Modelo B faz uso do hiperparâmetro **n_jobs=-1**, que instrui o algoritmo a utilizar todos os núcleos disponíveis do processador. Nesse caso, a paralelização ocorre em dois níveis: tanto na execução interna do próprio algoritmo quanto no processo de GridSearchCV, o que amplia o ganho de eficiência. Já o Modelo A aplica o **n_jobs=-1** apenas na etapa de treinamento do algoritmo em si, sem o mesmo benefício adicional no processo de busca de hiperparâmetros.

&nbsp;&nbsp;&nbsp;&nbsp;Essa diferença explica por que ambos os modelos apresentaram tempos de execução semelhantes, mesmo possuindo níveis de complexidade distintos, afinal, a Random Forest (Modelo B) é naturalmente mais custosa que uma única Árvore de Decisão (Modelo A), mas o aproveitamento mais amplo da paralelização reduz esse impacto.

### 4.5.1 Comparação por classes

&nbsp;&nbsp;&nbsp;&nbsp; Na seção anterior, verificamos que o modelo C apresentou médias superiores em todas as métricas avaliadas. Entretanto, isso não significa necessariamente que ele tenha o melhor desempenho em métricas específicas. Para identificar o modelo mais adequado em cada situação, realizamos o cálculo das métricas individualmente para cada classe. Os resultados estão apresentados a seguir:

| Modelo | Precisão Inadimplente | Precisão em dia | Precisão atrasado | Recall Inadimplente | Recall em dia | Recall atrasado | F1 Score Inadimplente | F1 Score em dia | F1 Score atrasado |
|--------|------------------------|-----------------|-------------------|---------------------|---------------|-----------------|-----------------------|-----------------|-------------------|
| A      | 70%                   | 38%             | 72%               | 66%                 | 47%           | 68%             | 68%                   | 42%             | 70%               |
| B      | 75%                   | 42%             | 69%               | 49%                 | 64%           | 76%             | 59%                   | 50%             | 73%               |
| C      | <span style="color:green; font-weight:bold">78%</span> | <span style="color:green; font-weight:bold">57%</span> | <span style="color:green; font-weight:bold">82%</span> | <span style="color:green; font-weight:bold">75%</span> | <span style="color:green; font-weight:bold">65%</span> | <span style="color:green; font-weight:bold">80%</span> | <span style="color:green; font-weight:bold">77%</span> | <span style="color:green; font-weight:bold">61%</span> | <span style="color:green; font-weight:bold">81%</span> |



&nbsp;&nbsp;&nbsp;&nbsp; A tabela mostra que o Modelo C apresenta o melhor desempenho geral, superando os demais em todas as métricas e em todas as classes. Isso indica que ele é mais consistente e equilibrado, entregando os maiores valores de precisão, recall e F1 Score independentemente da categoria avaliada. O Modelo A, por outro lado, tem desempenho mais fraco, especialmente na classe “em dia”, onde seus valores são significativamente inferiores aos dos demais, sugerindo dificuldade em classificar corretamente essa categoria. Já o Modelo B ocupa uma posição intermediária: em alguns casos consegue superar o Modelo A, mas ainda permanece abaixo do Modelo C em todos os aspectos.

### 4.5.2 Conclusões

&nbsp;&nbsp;&nbsp;&nbsp; Após a execução de todos os modelos, definimos como algoritmo definitivo para o modelo de tendência de pagamento o C (LightGBM), uma vez que apresentou desempenho superior em todas as métricas avaliadas, garantindo maior equilíbrio entre precisão, recall e F1 Score nas diferentes classes. Dessa forma, ele se mostra a escolha mais adequada para aplicação prática do sistema.

#### 4.5.2.2 Dificuldades

&nbsp;&nbsp;&nbsp;&nbsp; Apesar dos resultados, encontramos algumas dificuldades ao decorrer da sprint, sendo elas:

- Dificuldade na busca por hiperparâmetros pela falta de processamento dos nossos equipamentos.
- Complicações na instalação do Pycaret.
- Problemas em lidar com uma grande quantidade de dados.

#### 4.5.2.2 Planos Futuros

Definimos como planos futuros para o projeto:

- Utilizar o GridSearch no modelo C (Não foi possível durante essa essa Sprint por conta da falta de processamento dos nossos equipamentos)
- Melhorar a acurácia do modelo, principalmente para boletos **em dia**.
- Testar o modelo escolhido.
- Dedicar esforços a criação de um Dashboard funcional.



## 4.6 Preparando Modelo Preditivo de Valor de Recebimentos

Esta seção descreve o processo de construção de um modelo preditivo capaz de estimar se um boleto será pago integralmente, utilizando dados históricos de recebimentos. Esse modelo é uma ferramenta estratégica para o planejamento financeiro, pois permite antecipar fluxos de caixa e identificar padrões de comportamento dos pagadores.

### 4.6.1 Preparação dos Dados

A etapa de preparação dos dados é fundamental para assegurar a consistência, qualidade e representatividade das informações antes da modelagem.

#### 4.6.1.1 Conversão de Datas

As colunas de data (data_vencto, data_inclusao e dt_pagto) foram convertidas para o tipo datetime. Essa transformação é necessária, pois os dados originais podem estar armazenados como strings, o que inviabilizaria operações de natureza temporal.

Com a padronização para o formato datetime, torna-se possível realizar análises e cálculos como:

- Determinação dos intervalos entre inclusão, vencimento e pagamento;
- Identificação de atrasos ou adiantamentos;
- Criação de variáveis temporais derivadas, como o mês de vencimento ou o dia da semana do pagamento.


In [ ]:
TabelaGeral['data_vencto'] = pd.to_datetime(TabelaGeral['data_vencto'], errors='coerce', dayfirst=True)
TabelaGeral['data_inclusao'] = pd.to_datetime(TabelaGeral['data_inclusao'], errors='coerce', dayfirst=True)
TabelaGeral['dt_pagto'] = pd.to_datetime(TabelaGeral['dt_pagto'], errors='coerce', dayfirst=True)

#### 4.6.1.2 Cálculo de Dias para Pagamento e Recebimento em Dia
Foi criada a coluna diasNecessáriosParaPagar, responsável por calcular o número de dias entre a data de vencimento e a data de pagamento de cada boleto. A partir dessa métrica, gerou-se a variável binária recebe_em_dia, que assume o valor 1 quando o pagamento é realizado até 30 dias após o vencimento, e 0 caso contrário.

Essa variável é particularmente relevante, pois captura um comportamento temporal do pagador, refletindo sua tendência em realizar os pagamentos dentro de um prazo considerado aceitável. Esse tipo de informação contribui para aprimorar a capacidade preditiva do modelo em relação à pontualidade e ao risco de atraso.

In [ ]:
TabelaAtual['diasNecessáriosParaPagar'] = (TabelaAtual['dt_pagto'] - TabelaAtual['data_vencto']).dt.days

# Criando a coluna one-hot: 1 se <=30 dias, 0 caso contrário
TabelaAtual['recebe_em_dia'] = (TabelaAtual['diasNecessáriosParaPagar'] <= 30).astype(int)

# Visualizando o resultado
print(TabelaAtual[['diasNecessáriosParaPagar', 'recebe_em_dia']])

#### 4.6.1.3 Verificação de Pagamento Integral
Foi criada a coluna vl_bol_int, que representa a variável alvo do modelo. Essa coluna assume o valor 1 quando o valor efetivamente pago (vl_pagto) é igual ao valor original do boleto (vl_boleto), e 0 caso contrário.

Essa definição é fundamental, pois estabelece o foco da modelagem preditiva: identificar a probabilidade de um boleto ser pago integralmente. A partir dessa variável, o modelo é capaz de distinguir entre comportamentos de pagamento total e parcial, permitindo análises mais precisas sobre a adimplência dos pagadores.

In [ ]:
TabelaAtual['vl_bol_int'] = (TabelaAtual['vl_boleto'] == TabelaAtual['vl_pagto']).astype(int)
print(TabelaAtual['vl_bol_int'])

#### 4.6.1.4 Ajuste de Colunas
Nessa etapa, são realizadas verificações e transformações adicionais para assegurar que os dados estejam adequadamente formatados para a modelagem. Entre os principais procedimentos adotados, destacam-se:

- Substituição de valores nulos em colunas numéricas, evitando inconsistências durante o treinamento do modelo;
- Conversão de variáveis categóricas em representações numéricas, quando necessário, permitindo que o algoritmo interprete corretamente essas informações;
- Normalização de variáveis com escalas muito distintas, a fim de garantir que nenhuma feature exerça influência desproporcional sobre o processo de aprendizado.

Esses ajustes são essenciais para garantir a qualidade e compatibilidade dos dados, resultando em um modelo mais estável e preciso.

In [ ]:
TabelaAtual['vl_bol_int'] = TabelaAtual['vl_bol_int']
TabelaAtual['qtd_acesso_n'] = (TabelaAtual['qtd_acessos_pagador'] != "\\N")

## 4.6.2 Features do Modelo

As variáveis selecionadas exercem papel fundamental na capacidade do modelo de identificar padrões e relacionamentos relevantes nos dados.

A variável alvo (y) é vl_bol_int, que indica se o boleto foi pago integralmente (1) ou não (0).

As variáveis preditivas (X) utilizadas nesta versão são:

- vl_boleto: valor original do boleto;
- vl_pagto: valor efetivamente pago.

Embora essa configuração inicial seja simples, ela foi escolhida para garantir robustez e reduzir o risco de overfitting, permitindo uma avaliação mais controlada do desempenho do modelo em sua forma mais básica.

Em versões anteriores, também foram consideradas features adicionais, tais como:

- vl_boleto_norm e vl_pagto_norm: valores normalizados do boleto e do pagamento;
- juros_norm e multa_norm: indicadores de penalidades aplicadas;
- qtd_acessos_pagador_norm: número de acessos do pagador, normalizado;
- recebe_em_dia: indicador binário que sinaliza se o pagamento foi realizado até 30 dias após o vencimento.

Essas variáveis possuem potencial para enriquecer o modelo em versões futuras, aumentando sua capacidade de generalização. No entanto, foram omitidas nesta versão para evitar complexidade desnecessária e focar na validação do comportamento preditivo básico.

## 4.6.3 Divisão dos Dados
Para garantir uma avaliação imparcial do desempenho do modelo, os dados foram divididos em dois subconjuntos: treino e teste.

O conjunto de treino é utilizado para ajustar os parâmetros do modelo e realizar o aprendizado sobre os padrões existentes nos dados. Já o conjunto de teste é empregado exclusivamente na avaliação final, simulando o desempenho do modelo diante de novas observações não vistas anteriormente.

A divisão adotada foi de 70% para teste e 30% para treino, configuração menos comum — já que o padrão usual é o inverso —, mas justificada possivelmente pelo tamanho reduzido do dataset ou pela necessidade de avaliar o modelo em uma amostra de teste mais ampla. Essa decisão visa obter uma estimativa mais robusta da capacidade de generalização do modelo.

### 4.6.4 Modelo A - Regressão Logística

O primeiro modelo implementado foi a Regressão Logística, amplamente utilizada para problemas de classificação binária, como a previsão de pagamento integral ou não de boletos.

**Justificativa da escolha:**

A escolha desse modelo se baseia em sua simplicidade de interpretação e eficiência em cenários lineares, além de possibilitar a análise direta da importância das variáveis preditoras. Trata-se de um modelo estatístico consolidado, adequado para uma primeira abordagem exploratória do problema.

**Configuração do modelo:**

- class_weight='balanced': ajusta automaticamente os pesos das classes, compensando eventuais desbalanceamentos entre boletos pagos integralmente e não pagos.
- solver='sag': método de otimização eficiente para conjuntos de dados grandes e esparsos.

Também foi testada uma variação com pesos iguais entre as classes, permitindo comparar o impacto do balanceamento automático no desempenho geral do modelo.

In [ ]:
X = TabelaAtual[['vl_boleto', 'vl_pagto', ]]
y = TabelaAtual['vl_bol_int']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.7, random_state=42)

# Logistic Regression model
model = LogisticRegression(class_weight='balanced', solver='sag')
model = LogisticRegression(class_weight={0: 0.5, 1: 0.5}, solver='sag')
# model = RandomForestClassifier()
model.fit(X_train, y_train)

### 4.6.5 Avalição do Modelo
Após o treinamento, o modelo é avaliado sobre o conjunto de teste, permitindo medir a sua capacidade de generalização. O objetivo é verificar se o modelo não apenas aprendeu os padrões dos dados de treino, mas também consegue fazer previsões consistentes para novos boletos.

### 4.6.6 Métricas de avaliação

Para avaliar o desempenho dos modelos de classificação, foram adotadas métricas amplamente utilizadas em problemas binários, cada uma oferecendo uma perspectiva complementar sobre a qualidade das previsões.

Acurácia (Accuracy):
Representa a proporção de previsões corretas em relação ao total de observações.
Embora de fácil interpretação, pode ser enganosa em bases desbalanceadas, nas quais uma classe é muito mais frequente que a outra.

Precisão (Precision):
Indica, entre os boletos previstos como pagos integralmente, quantos realmente foram pagos.
É especialmente útil para reduzir falsos positivos, ou seja, evitar superestimar o número de pagamentos integrais.

Recall (Sensibilidade):
Mede, entre os boletos que foram de fato pagos integralmente, quantos foram corretamente identificados pelo modelo.
Essa métrica é importante para não deixar de reconhecer casos positivos reais, garantindo maior cobertura.

F1-score:
Corresponde à média harmônica entre precisão e recall, equilibrando ambas as métricas em um único indicador de desempenho global.

ROC AUC (Área sob a curva ROC):
Avalia a capacidade do modelo de distinguir entre boletos pagos integralmente (classe 1) e não pagos integralmente (classe 0).
Quanto mais próximo de 1, melhor o poder discriminativo do modelo.

Matriz de Confusão:
Apresenta a distribuição de acertos e erros em cada classe, permitindo identificar tendências do modelo, como uma possível propensão a superestimar ou subestimar pagamentos integrais.

In [ ]:
predicaoBoleto = model.predict(X_test)

y_pred_proba = model.predict_proba(X_test)[:, 1]

print(predicaoBoleto)
print(y_pred_proba)

print("Acurácia:", accuracy_score(y_test, predicaoBoleto))
print("ROC AUC:", roc_auc_score(y_test, y_pred_proba))
print("\nRelatório de classificação:\n", classification_report(y_test, predicaoBoleto))
print(confusion_matrix(y_test, predicaoBoleto))

# Torna as variáveis principais deste modelo acessíveis globalmente
import inspect

# Coleta todas as variáveis locais da célula atual
local_vars = dict(inspect.currentframe().f_locals)

# Filtra só as variáveis úteis (evita funções, módulos e dados muito grandes)
variaveis_interessantes = {
    k: v for k, v in local_vars.items()
    if not k.startswith("__") and not callable(v) and not isinstance(v, type(inspect))
}

# Atualiza o escopo global com as variáveis dessa célula
globals().update(variaveis_interessantes)

print(f" Variáveis exportadas para o escopo global: {list(variaveis_interessantes.keys())}")

### 4.6.7 Aplicação do Modelo

Após a etapa de validação, o modelo é aplicado ao conjunto de teste para estimar, para cada boleto, a probabilidade de pagamento integral (proba_int_pag).

Essas probabilidades permitem converter a classificação binária (pago integralmente ou não) em uma escala contínua, representando o grau de confiança do modelo em cada previsão.

Em seguida, é realizado o agrupamento por pagador, calculando-se a média das probabilidades individuais.
Esse processo fornece uma visão consolidada do comportamento histórico e preditivo de cada cliente, auxiliando na identificação de perfis com maior ou menor propensão ao pagamento integral.

In [ ]:
# Adicionar coluna apenas nas linhas do teste
TabelaAtual.loc[X_test.index, 'proba_int_pag'] = y_pred_proba

resultado = (
    TabelaAtual.loc[X_test.index]
    .groupby('pagador_inscricao_hash')['proba_int_pag']
    .mean()  # média das probabilidades por pagador
    .reset_index()
)

print(resultado)

#### 4.4.9 Cálculo do Pagamento Total Previsto por Pagador
Após o cálculo das probabilidades de pagamento integral, realiza-se a transformação dos resultados probabilísticos em valores monetários previstos, permitindo estimar o montante financeiro esperado por pagador.

Primeiramente, são obtidos, para cada pagador:

- soma_valor_boleto – soma total dos valores originais dos boletos;
- soma_vl_pagto – soma total dos pagamentos efetivamente realizados;
- diferenca – média histórica da diferença entre o valor do boleto e o valor pago.

Com base nessas informações, aplica-se a seguinte lógica de previsão:

- Se a probabilidade média de pagamento integral > 0.5 → assume-se que o pagador quitará 100% do valor total dos boletos;
- Caso contrário → o valor previsto é ajustado subtraindo-se a diferença média histórica do total.

Essa abordagem combina a interpretação probabilística do modelo com uma análise comportamental histórica, permitindo estimar com maior realismo os valores esperados de pagamento.

In [ ]:
# Agrupa e calcula soma dos valores para valor_boleto e vl_pagto e quantidade por pagador no conjunto de teste
agrupado = TabelaAtual.loc[X_test.index].groupby('pagador_inscricao_hash').agg(
    soma_valor_boleto=('vl_boleto', 'sum'),
    soma_vl_pagto=('vl_pagto', 'sum'),
    contagem=('vl_boleto', 'count')
)

# Calcula a diferença para cada pagador
agrupado['diferenca'] = (agrupado['soma_valor_boleto'] - agrupado['soma_vl_pagto']) / agrupado['contagem']

# Inicializa a lista que vai guardar o resultado da condição para cada linha do resultado
pagamento_total_lista = []

# Loop para iterar cada linha do resultado
for row in resultado.itertuples(index=False):
    pagador = row.pagador_inscricao_hash
    proba = row.proba_int_pag

# Pega a soma do valor_boleto e a diferença para o pagador, ou 0 se não existir



    soma_boleto = agrupado['soma_valor_boleto'].get(pagador, 0)
    diff = agrupado['diferenca'].get(pagador, 0)

    if proba > 0.5:
        pagamento_total_lista.append(soma_boleto)
    else:
        pagamento_total_lista.append(soma_boleto - diff)

# Cria a nova coluna no DataFrame resultado
resultado['pagamento_total'] = pagamento_total_lista


### 4.6.10 Comparação dos Valores Totais
Por fim, realiza-se a comparação entre os valores previstos pelo modelo e os valores reais observados, permitindo avaliar a acurácia das estimativas financeiras geradas. Essa análise é realizada em dois níveis:

- Por Pagador: compara-se o valor total de boletos emitidos com o valor total previsto de pagamento, fornecendo uma visão individual do desempenho do modelo para cada cliente.

- Total Geral: compara-se a soma de todos os boletos com a soma dos valores previstos, avaliando a capacidade do modelo de estimar corretamente a receita total esperada.

Além disso, são calculadas métricas de erro para quantificar a precisão das previsões:

- MAE (Erro Médio Absoluto): mede a diferença média absoluta entre os valores reais e previstos, oferecendo uma interpretação direta do erro médio.

- RMSE (Raiz do Erro Quadrático Médio): penaliza mais fortemente os erros maiores, fornecendo uma visão mais sensível a outliers e a variações significativas nos pagamentos.

Essa abordagem permite não apenas avaliar a capacidade preditiva do modelo, mas também identificar possíveis distorções ou limitações na previsão financeira tanto em nível individual quanto agregado.

In [ ]:
# Mostrar comparação por pagador (valor_boleto total vs pagamento_total)

comparacao_pagador = pd.DataFrame({
    'valor_boleto_total': agrupado['soma_valor_boleto'],
    'pagamento_total_previsto': resultado.set_index('pagador_inscricao_hash')['pagamento_total']
}).dropna()

print("\nComparação por pagador:")
print(comparacao_pagador.head())

# Comparação total geral

total_valor_boleto = agrupado['soma_valor_boleto'].sum()
total_pagamento_previsto = resultado['pagamento_total'].sum()

print(f"\nTotal valor_boleto: {total_valor_boleto:.2f}")
print(f"Total pagamento previsto: {total_pagamento_previsto:.2f}")
print(f"Porcentagem do total previsto em relação ao valor_boleto: {(total_pagamento_previsto / total_valor_boleto) * 100:.2f}%")

# Torna as variáveis principais deste modelo acessíveis globalmente
import inspect

# Coleta todas as variáveis locais da célula atual
local_vars = dict(inspect.currentframe().f_locals)

# Filtra só as variáveis úteis (evita funções, módulos e dados muito grandes)
variaveis_interessantes = {
    k: v for k, v in local_vars.items()
    if not k.startswith("__") and not callable(v) and not isinstance(v, type(inspect))
}

# Atualiza o escopo global com as variáveis dessa célula
globals().update(variaveis_interessantes)

print(f" Variáveis exportadas para o escopo global: {list(variaveis_interessantes.keys())}")

In [ ]:
# Criar coluna de previsão baseada na lógica do modelo
agrupado['previsto'] = agrupado['soma_valor_boleto'] - agrupado['diferenca']

mae = mean_absolute_error(agrupado['soma_vl_pagto'], agrupado['previsto'])
rmse = np.sqrt(mean_squared_error(agrupado['soma_vl_pagto'], agrupado['previsto']))

# Torna as variáveis principais deste modelo acessíveis globalmente
import inspect

# Coleta todas as variáveis locais da célula atual
local_vars = dict(inspect.currentframe().f_locals)

# Filtra só as variáveis úteis (evita funções, módulos e dados muito grandes)
variaveis_interessantes = {
    k: v for k, v in local_vars.items()
    if not k.startswith("__") and not callable(v) and not isinstance(v, type(inspect))
}

# Atualiza o escopo global com as variáveis dessa célula
globals().update(variaveis_interessantes)

print(f" Variáveis exportadas para o escopo global: {list(variaveis_interessantes.keys())}")

print(f'MAE (Erro Médio Absoluto): {mae:.2f}')
print(f'RMSE (Raiz do Erro Quadrático Médio): {rmse:.2f}')

# 4.7 Preparando Modelo Preditivo de Valor de Recebimentos - Random Forest

Nesta seção, foi implementada uma abordagem baseada em Random Forest (árvore aleatória) para prever a probabilidade de pagamento integral dos boletos.

A escolha do Random Forest se justifica por sua capacidade de:

- Capturar relações não lineares entre as variáveis;

- Lidar eficientemente com features correlacionadas;

- Reduzir o risco de overfitting em comparação a árvores de decisão individuais, mantendo a robustez do modelo.

Essa abordagem permite gerar previsões mais confiáveis e detalhadas, mantendo equilíbrio entre complexidade e interpretabilidade.

### 4.7.1 Treinamento do Modelo

O modelo foi treinado com o objetivo de prever a probabilidade de pagamento integral dos boletos (vl_bol_int), utilizando as features selecionadas (vl_boleto e vl_pagto).

Durante o treinamento, o algoritmo aprende padrões e relações entre os valores dos boletos e os pagamentos efetivos, permitindo estimar de forma probabilística a probabilidade de cada boleto ser quitado integralmente.

### 4.7.2 Divisão dos Dados
Para garantir uma avaliação imparcial do modelo, os dados foram divididos em conjuntos de treino e teste, respeitando a seguinte configuração:

- Proporção: 30% para treino e 70% para teste;

- Random State = 42: assegura a reprodutibilidade dos resultados;

- Estratificação: mantém a proporção da variável alvo (vl_bol_int) em ambos os conjuntos, evitando desbalanceamento na divisão e garantindo representatividade adequada das classes.

Essa divisão permite que o modelo seja treinado em uma amostra consistente e avaliado de forma confiável em dados não vistos anteriormente.

### 4.7.3 Modelo B - Random Forest

O Modelo B foi implementado utilizando o algoritmo Random Forest Classifier, que combina múltiplas árvores de decisão para melhorar a robustez e reduzir o risco de overfitting.

**Hiperparâmetros configurados:**

- n_estimators = 100 – número de árvores na floresta;

- max_depth = 10 – profundidade máxima de cada árvore;

- min_samples_split = 5 – número mínimo de amostras para dividir um nó;

- min_samples_leaf = 2 – número mínimo de amostras em um nó folha;

- class_weight = 'balanced' – ajuste automático dos pesos das classes para lidar com desbalanceamento.

A otimização dos hiperparâmetros foi realizada utilizando RandomizedSearchCV, garantindo uma configuração eficiente e balanceada entre desempenho e complexidade.

O objetivo principal das métricas aplicadas é comparar o desempenho do Modelo B com o Modelo A (Regressão Logística). As métricas consideradas incluem:

- Acurácia: avalia o desempenho global do modelo;

- ROC AUC: mede a capacidade de separar corretamente as classes, mesmo em dados desbalanceados;

- Relatório de Classificação: detalha precisão, recall e F1-score por classe, oferecendo uma análise completa do desempenho em cada categoria da variável alvo.

In [ ]:
X2 = TabelaAtual[['vl_boleto', 'vl_pagto', ]]
y2 = TabelaAtual['vl_bol_int']

# Train/test split
X_train2, X_test2, y_train2, y_test2 = train_test_split(X2, y2, test_size=0.7, random_state=42)

# Random Forest model
model2 = RandomForestClassifier(class_weight='balanced', n_estimators=100, random_state=42, max_depth=10, min_samples_split=5, min_samples_leaf=2)
model2.fit(X_train2, y_train2)

### 4.7.4 Avaliação do Modelo
Após o treinamento, o Modelo B (Random Forest) foi avaliado utilizando o conjunto de teste, a fim de verificar sua capacidade de generalização em dados não vistos.

As métricas principais aplicadas foram:

- Acurácia: mede a proporção de previsões corretas em relação ao total de boletos;

- ROC AUC: avalia a habilidade do modelo em discriminar corretamente entre boletos pagos integralmente (classe 1) e não pagos integralmente (classe 0);

- Relatório de Classificação: detalha precisão, recall e F1-score por classe, oferecendo uma visão granular do desempenho;

- Matriz de Confusão: permite visualizar a distribuição de acertos e erros em cada classe, facilitando a identificação de possíveis vieses do modelo.

Essa avaliação fornece uma visão completa do desempenho do modelo, tanto em termos globais quanto em relação às classes específicas da variável alvo.

In [ ]:
predicaoBoleto2 = model2.predict(X_test2)

y_pred_proba2 = model2.predict_proba(X_test2)[:, 1]

print(predicaoBoleto2)
print(y_pred_proba2)

print("Acurácia:", accuracy_score(y_test2, predicaoBoleto2))
print("ROC AUC:", roc_auc_score(y_test2, y_pred_proba2))
print("\nRelatório de classificação:\n", classification_report(y_test2, predicaoBoleto2))
print(confusion_matrix(y_test2, predicaoBoleto2))

# Coleta todas as variáveis locais da célula atual
local_vars = dict(inspect.currentframe().f_locals)

# Filtra só as variáveis úteis (evita funções, módulos e dados muito grandes)
variaveis_interessantes = {
    k: v for k, v in local_vars.items()
    if not k.startswith("__") and not callable(v) and not isinstance(v, type(inspect))
}

# Atualiza o escopo global com as variáveis dessa célula
globals().update(variaveis_interessantes)

print(f" Variáveis exportadas para o escopo global: {list(variaveis_interessantes.keys())}")

### 4.7.5 Aplicação do Modelo

Após o treinamento e validação, o Modelo B (Random Forest) é aplicado ao conjunto de teste para estimar, para cada boleto, a probabilidade de pagamento integral (proba_int_pag).

Em seguida, essas probabilidades são agregadas por pagador, calculando-se a média das probabilidades individuais. Essa abordagem fornece uma visão consolidada do comportamento esperado de cada cliente, permitindo identificar padrões de adimplência e perfis com maior ou menor propensão ao pagamento integral.

In [ ]:
# Adicionar coluna apenas nas linhas do teste
TabelaAtual.loc[X_test2.index, 'proba_int_pag'] = y_pred_proba2

resultado2 = (
    TabelaAtual.loc[X_test.index]
    .groupby('pagador_inscricao_hash')['proba_int_pag']
    .mean()  # média das probabilidades por pagador
    .reset_index()
)

print(resultado)

### 4.7.6 Cálculo do Pagamento Total Previsto por Pagador
Após a agregação das probabilidades, é realizado o cálculo do valor total previsto de pagamento para cada pagador, ajustando as estimativas com base na probabilidade média de pagamento integral.

O procedimento segue os seguintes passos:

1. Agrupamento por pagador: são somados os valores originais dos boletos (soma_valor_boleto) e os pagamentos efetivamente realizados (soma_vl_pagto), além de registrar a contagem de boletos por cliente.

2. Cálculo da diferença média histórica (diferenca): mede a média das discrepâncias entre os valores dos boletos e os pagamentos efetivos, capturando o comportamento típico do pagador.

3. Aplicação da lógica de previsão:
    - Se a probabilidade média de pagamento integral > 0,5 → assume-se que o pagador quitará 100% do valor total dos boletos;
    - Caso contrário → o valor previsto é ajustado subtraindo a diferença média histórica, refletindo o padrão de pagamento parcial observado.

Essa abordagem combina a interpretação probabilística do modelo com informações comportamentais históricas, permitindo estimativas financeiras mais realistas e confiáveis.

In [ ]:
# Agrupa e calcula soma dos valores para valor_boleto e vl_pagto e quantidade por pagador no conjunto de teste
agrupado2 = TabelaAtual.loc[X_test2.index].groupby('pagador_inscricao_hash').agg(
    soma_valor_boleto=('vl_boleto', 'sum'),
    soma_vl_pagto=('vl_pagto', 'sum'),
    contagem=('vl_boleto', 'count')
)

# Calcula a diferença para cada pagador
agrupado2['diferenca'] = (agrupado2['soma_valor_boleto'] - agrupado2['soma_vl_pagto']) / agrupado2['contagem']

# Inicializa a lista que vai guardar o resultado da condição para cada linha do resultado
pagamento_total_lista2 = []

# Loop para iterar cada linha do resultado
for row in resultado2.itertuples(index=False):
    pagador = row.pagador_inscricao_hash
    proba = row.proba_int_pag

# Pega a soma do valor_boleto e a diferença para o pagador, ou 0 se não existir



    soma_boleto2 = agrupado2['soma_valor_boleto'].get(pagador, 0)
    diff = agrupado2['diferenca'].get(pagador, 0)

    if proba > 0.5:
        pagamento_total_lista2.append(soma_boleto2)
    else:
        pagamento_total_lista2.append(soma_boleto2 - diff)

# Cria a nova coluna no DataFrame resultado2
resultado2['pagamento_total'] = pagamento_total_lista2

### 4.7.7 Comparação dos Valores Totais
Por fim, realiza-se a comparação entre os valores previstos pelo modelo e os valores reais observados, permitindo avaliar a precisão das estimativas financeiras geradas pelo Random Forest.

Essa análise é realizada em dois níveis:

- Por Pagador: compara o valor total de boletos emitidos com o valor total previsto de pagamento, fornecendo uma visão individual do desempenho do modelo para cada cliente;

- Agregado (Total Geral): compara a soma de todos os boletos com a soma dos valores previstos, avaliando a capacidade do modelo de estimar corretamente a receita total esperada.

Além disso, são calculadas métricas de erro para quantificar a precisão das previsões:

- MAE (Erro Médio Absoluto): diferença média absoluta entre os valores reais e previstos;

- RMSE (Raiz do Erro Quadrático Médio): penaliza erros maiores de forma mais intensa, fornecendo uma análise mais sensível a outliers e variações significativas nos pagamentos.

Essa abordagem permite avaliar tanto a acurácia global do modelo quanto sua capacidade de previsão em nível individual, fornecendo insights valiosos para tomada de decisão financeira.

In [ ]:
# Mostrar comparação por pagador (valor_boleto total vs pagamento_total)

comparacao_pagador2 = pd.DataFrame({
    'valor_boleto_total': agrupado2['soma_valor_boleto'],
    'pagamento_total_previsto': resultado2.set_index('pagador_inscricao_hash')['pagamento_total']
}).dropna()

print("\nComparação por pagador:")
print(comparacao_pagador2.head())

# Comparação total geral

total_valor_boleto2 = agrupado2['soma_valor_boleto'].sum()
total_pagamento_previsto2 = resultado2['pagamento_total'].sum()

print(f"\nTotal valor_boleto: {total_valor_boleto2:.2f}")
print(f"Total pagamento previsto: {total_pagamento_previsto2:.2f}")
print(f"Porcentagem do total previsto em relação ao valor_boleto: {(total_pagamento_previsto2 / total_valor_boleto2) * 100:.2f}%")

# Torna as variáveis principais deste modelo acessíveis globalmente
import inspect

# Coleta todas as variáveis locais da célula atual
local_vars = dict(inspect.currentframe().f_locals)

# Filtra só as variáveis úteis (evita funções, módulos e dados muito grandes)
variaveis_interessantes = {
    k: v for k, v in local_vars.items()
    if not k.startswith("__") and not callable(v) and not isinstance(v, type(inspect))
}

# Atualiza o escopo global com as variáveis dessa célula
globals().update(variaveis_interessantes)

print(f" Variáveis exportadas para o escopo global: {list(variaveis_interessantes.keys())}")

In [ ]:
# Criar coluna de previsão baseada na lógica do modelo
agrupado2['previsto'] = agrupado2['soma_valor_boleto'] - agrupado2['diferenca']

mae2 = mean_absolute_error(agrupado2['soma_vl_pagto'], agrupado2['previsto'])
rmse2 = np.sqrt(mean_squared_error(agrupado2['soma_vl_pagto'], agrupado2['previsto']))

# Torna as variáveis principais deste modelo acessíveis globalmente
import inspect

# Coleta todas as variáveis locais da célula atual
local_vars = dict(inspect.currentframe().f_locals)

# Filtra só as variáveis úteis (evita funções, módulos e dados muito grandes)
variaveis_interessantes = {
    k: v for k, v in local_vars.items()
    if not k.startswith("__") and not callable(v) and not isinstance(v, type(inspect))
}

# Atualiza o escopo global com as variáveis dessa célula
globals().update(variaveis_interessantes)

print(f" Variáveis exportadas para o escopo global: {list(variaveis_interessantes.keys())}")

print(f'MAE (Erro Médio Absoluto): {mae2:.2f}')
print(f'RMSE (Raiz do Erro Quadrático Médio): {rmse2:.2f}')

## 4.8 Modelo Preditivo de Recebimento de Pagamentos 3

### 4.8.1 Troca de features

Os modelos anteriores utilizavam a coluna "vl_pagto" no treinamento. No entanto, isso fazia com que o modelo aprendesse a partir de uma informação futura, inviabilizando a avaliação correta de novos boletos, já que esses ainda não possuem valor de pagamento registrado. Para corrigir essa limitação, foi realizada uma alteração nas features utilizadas.

In [ ]:
X3 = TabelaAtual[['vl_boleto', 'pagador_cnpjcpf_norm', 'dia_vencimento_norm', 'mes_vencimento_norm', 'qtd_acesso_n']]
y3 = TabelaAtual['vl_bol_int']

# Train/test split
X_train3, X_test3, y_train3, y_test3 = train_test_split(X3, y3, test_size=0.7, random_state=42)

### 4.8.2 Procurar o melhor algortmo usando Pycaret

Para identificar o algoritmo com melhor desempenho a partir das features selecionadas, foi utilizada a biblioteca PyCaret, que automatiza a busca, o teste e a comparação entre diversos algoritmos de machine learning.

**Observação**: O código PyCaret foi comentado no notebook devido a conflitos de versões do Python necessárias para a instalação de algumas dependências. Apesar disso, a documentação registra a intenção de sua utilização para automatizar a experimentação de modelos, servindo como referência para implementações futuras em ambientes com compatibilidade de versões.

In [ ]:
# from pycaret.regression import setup, compare_models

# grid = setup(data=X3, target=y3, html=False, verbose=False, fold=2, fold_strategy='stratifiedkfold')
# best_models = compare_models()
# print(best_models)

### 4.8.3 Criação do modelo usando Light Gradient Boosting Machine

De acordo com os resultados obtidos com o PyCaret, o algoritmo Light Gradient Boosting Machine (LightGBM) apresentou o melhor recall para a classe negativa, a maior precisão para a classe positiva e a melhor acurácia geral do modelo. Esses resultados justificam a escolha desse algoritmo para a etapa de comparação.

#### 4.8.3.1 Treinamento do modelo

Inicialmente, o modelo foi treinado utilizando os hiperparâmetros padrão, sem ajustes específicos, permitindo uma avaliação preliminar do desempenho antes da aplicação de técnicas de otimização e calibração de parâmetros.

In [ ]:
from lightgbm import LGBMClassifier

model3 = LGBMClassifier(
    n_estimators=100,     # número de árvores
    learning_rate=0.1,    # taxa de aprendizado
    max_depth=-1,         # profundidade (ou ilimitado se -1)
    random_state=42
)

model3.fit(X_train3, y_train3)

#### 4.8.3.2 Teste do modelo

Com o modelo já treinado, foi conduzida uma avaliação de teste, a qual evidenciou melhorias significativas em relação ao modelo original, tanto em termos de métricas de classificação quanto na capacidade de previsão de valores monetários, conforme demonstrado a seguir.

In [ ]:
predicaoBoleto3 = model3.predict(X_test3)

y_pred_proba3 = model3.predict_proba(X_test3)[:, 1]

print(predicaoBoleto3)
print(y_pred_proba3)

print("Acurácia:", accuracy_score(y_test3, predicaoBoleto3))
print("ROC AUC:", roc_auc_score(y_test3, y_pred_proba3))
print("\nRelatório de classificação:\n", classification_report(y_test3, predicaoBoleto3))
print("\nMatriz de confusão:\n", confusion_matrix(y_test3, predicaoBoleto3))

# Torna as variáveis principais deste modelo acessíveis globalmente
import inspect

# Coleta todas as variáveis locais da célula atual
local_vars = dict(inspect.currentframe().f_locals)

# Filtra só as variáveis úteis (evita funções, módulos e dados muito grandes)
variaveis_interessantes = {
    k: v for k, v in local_vars.items()
    if not k.startswith("__") and not callable(v) and not isinstance(v, type(inspect))
}

# Atualiza o escopo global com as variáveis dessa célula
globals().update(variaveis_interessantes)

print(f" Variáveis exportadas para o escopo global: {list(variaveis_interessantes.keys())}")

### 4.8.4 Melhores hiperparâmetros

Com o objetivo de otimizar o desempenho do modelo e aprimorar suas métricas de avaliação, foi realizado um Grid Search, permitindo a identificação da combinação ideal de hiperparâmetros para maximizar a precisão, a capacidade de discriminação (ROC AUC) e a robustez do modelo frente a diferentes padrões nos dados.

In [ ]:
# Hiperparâmetros otimizados para LightGBM

param_dist = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1],
    'max_depth': [-1, 10],
    'num_leaves': [31, 50],
    'min_child_samples': [10, 20]
}


grid_search = GridSearchCV(
    estimator=model3,
    param_grid=param_dist,
    scoring='accuracy',
    cv=3,
    n_jobs=1
)

grid_search.fit(X_train3, y_train3)
best_model = grid_search.best_estimator_
print("Melhores parâmetros:", grid_search.best_params_)

### 4.8.5 Modelo final

#### 4.8.5.1 Treinamento do modelo

Após a identificação da combinação ótima de hiperparâmetros, o modelo foi reimplementado e treinado novamente, incorporando os ajustes encontrados para maximizar seu desempenho e melhorar a acurácia e capacidade de previsão em dados não vistos.

In [ ]:
model3 = LGBMClassifier(
    n_estimators=100,        # número de árvores
    learning_rate=0.1,       # taxa de aprendizado
    max_depth=10,             # veio do grid search
    num_leaves=31,          # já estava no seu modelo
    min_child_samples=10,     # min_samples_leaf do grid search
    random_state=42
)

model3.fit(X_train3, y_train3)

#### 4.8.5.2 Teste do modelo

Após o treinamento, o modelo ajustado com os novos hiperparâmetros foi submetido a testes de avaliação, permitindo mensurar seu desempenho em relação à versão anterior e verificar melhorias nas métricas de classificação e previsão financeira.

In [ ]:
y_pred = model3.predict(X_test3)

print("Acurácia:", accuracy_score(y_test3, y_pred))
print("\nRelatório de classificação:\n", classification_report(y_test2, y_pred))
print("\nMatriz de confusão:\n", confusion_matrix(y_test2, y_pred))

# Torna as variáveis principais deste modelo acessíveis globalmente
import inspect

# Coleta todas as variáveis locais da célula atual
local_vars = dict(inspect.currentframe().f_locals)

# Filtra só as variáveis úteis (evita funções, módulos e dados muito grandes)
variaveis_interessantes = {
    k: v for k, v in local_vars.items()
    if not k.startswith("__") and not callable(v) and not isinstance(v, type(inspect))
}

# Atualiza o escopo global com as variáveis dessa célula
globals().update(variaveis_interessantes)

print(f" Variáveis exportadas para o escopo global: {list(variaveis_interessantes.keys())}")

Ao final desta etapa, observa-se que o modelo final apresenta desempenho superior em comparação ao modelo anterior, destacando-se nos seguintes aspectos:

- Recall aprimorado para a classe negativa, indicando maior capacidade de identificar corretamente boletos não pagos integralmente;

- Precisão superior para a classe positiva, refletindo menor incidência de falsos positivos na previsão de pagamentos integrais;

- Acurácia geral aumentada, evidenciando melhor desempenho global do modelo;

- F1-score mais elevado para a classe negativa, demonstrando equilíbrio entre precisão e recall nessa categoria.

Esses resultados confirmam que a otimização dos hiperparâmetros contribuiu significativamente para melhorar a robustez e a confiabilidade das previsões do modelo.

### 4.8.6 Aplicação do modelo

Nesta etapa, o modelo treinado é aplicado ao conjunto de teste para estimar o comportamento de pagamento dos boletos e gerar previsões financeiras.

#### 4.8.6.1 Probabilidade de pagamento por pagador

Nesta etapa, o modelo calcula, para cada pagador no conjunto de teste, a probabilidade média de pagamento integral.

Esse cálculo permite transformar as previsões individuais de cada boleto em uma estimativa consolidada do comportamento de pagamento do cliente, oferecendo uma visão agregada da propensão de cada pagador a quitar integralmente seus boletos.

In [ ]:

TabelaAtual.loc[X_test3.index, 'proba_int_pag'] = y_pred_proba3

resultado3 = (
    TabelaAtual.loc[X_test.index]
    .groupby('pagador_inscricao_hash')['proba_int_pag']
    .mean()  # média das probabilidades por pagador
    .reset_index()
)

print(resultado3)

#### 4.8.6.2 Cálculo de pagamento total previsto por pagador

Nesta etapa, calcula-se o valor total previsto de pagamento para cada pagador, considerando a probabilidade estimada de pagamento integral de cada boleto.

O procedimento consiste nos seguintes passos:

- Agrupamento por pagador: os boletos do conjunto de teste são agregados por cliente, permitindo consolidar as informações financeiras;

- Cálculo das somas: são computadas a soma dos valores originais dos boletos (valor_boleto), a soma dos pagamentos efetivamente realizados (vl_pagto) e a contagem de boletos por pagador.

Essa agregação fornece uma visão consolidada do comportamento financeiro esperado de cada cliente, permitindo estimar com precisão o montante total de pagamento previsto e facilitando análises subsequentes de receita.

In [ ]:
# Agrupa dados por pagador, calculando soma dos boletos, soma dos pagamentos e quantidade de boletos
agrupado3 = (
    TabelaAtual.loc[X_test3.index]
    .groupby('pagador_inscricao_hash')
    .agg(
        soma_valor_boleto=('vl_boleto', 'sum'),
        soma_vl_pagto=('vl_pagto', 'sum'),
        contagem=('vl_boleto', 'count')
    )
)

# Calcula a diferença média entre valor do boleto e valor pago por transação
agrupado3['diferenca'] = (
    (agrupado3['soma_valor_boleto'] - agrupado3['soma_vl_pagto'])
    / agrupado3['contagem']
)

# Inicializa lista que armazenará o valor total de pagamento estimado
pagamento_total_lista3 = []

# Itera sobre cada registro do DataFrame de resultados
for row in resultado3.itertuples(index=False):
    pagador = row.pagador_inscricao_hash
    proba = row.proba_int_pag

    # Recupera soma de boletos e diferença média para o pagador
    soma_boleto3 = agrupado3['soma_valor_boleto'].get(pagador, 0)
    diff = agrupado3['diferenca'].get(pagador, 0)

    # Aplica regra de decisão com base na probabilidade prevista de pagamento
    if proba > 0.5:
        pagamento_total_lista3.append(soma_boleto3)
    else:
        pagamento_total_lista3.append(soma_boleto3 - diff)

# Adiciona os valores calculados como nova coluna no DataFrame resultado3
resultado3['pagamento_total'] = pagamento_total_lista3

#### 4.8.6.3 Comparação dos valores totais
Nesta etapa, realiza-se a comparação entre os valores totais reais dos boletos e os valores totais previstos de pagamento, permitindo avaliar a acurácia das estimativas financeiras do modelo.

A análise é realizada em dois níveis:

- Por pagador: compara-se o valor total de boletos emitidos com o valor total previsto de pagamento para cada cliente, fornecendo uma visão individual do desempenho do modelo;

- Total agregado: compara-se a soma de todos os boletos com a soma dos valores previstos, avaliando a capacidade do modelo de estimar corretamente a receita total esperada.

Essa abordagem permite identificar tanto a precisão global do modelo quanto seu desempenho em nível individual, oferecendo insights importantes para tomada de decisão financeira.

In [ ]:
# Mostrar comparação por pagador (valor_boleto total vs pagamento_total)

comparacao_pagador3 = pd.DataFrame({
    'valor_boleto_total': agrupado3['soma_valor_boleto'],
    'pagamento_total_previsto': resultado3.set_index('pagador_inscricao_hash')['pagamento_total']
}).dropna()

print("\nComparação por pagador:")
print(comparacao_pagador3.head())

# Comparação total geral

total_valor_boleto3 = agrupado3['soma_valor_boleto'].sum()
total_pagamento_previsto3 = resultado3['pagamento_total'].sum()

print(f"\nTotal valor_boleto: {total_valor_boleto3:.2f}")
print(f"Total pagamento previsto: {total_pagamento_previsto3:.2f}")
print(f"Porcentagem do total previsto em relação ao valor_boleto: {(total_pagamento_previsto3 / total_valor_boleto3) * 100:.2f}%")

# Torna as variáveis principais deste modelo acessíveis globalmente
import inspect

# Coleta todas as variáveis locais da célula atual
local_vars = dict(inspect.currentframe().f_locals)

# Filtra só as variáveis úteis (evita funções, módulos e dados muito grandes)
variaveis_interessantes = {
    k: v for k, v in local_vars.items()
    if not k.startswith("__") and not callable(v) and not isinstance(v, type(inspect))
}

# Atualiza o escopo global com as variáveis dessa célula
globals().update(variaveis_interessantes)

print(f" Variáveis exportadas para o escopo global: {list(variaveis_interessantes.keys())}")

In [ ]:
# Criar coluna de previsão baseada na lógica do modelo
agrupado3['previsto'] = agrupado3['soma_valor_boleto'] - agrupado3['diferenca']

mae3 = mean_absolute_error(agrupado3['soma_vl_pagto'], agrupado3['previsto'])
rmse3 = np.sqrt(mean_squared_error(agrupado3['soma_vl_pagto'], agrupado3['previsto']))

print(f'MAE (Erro Médio Absoluto): {mae3:.2f}')
print(f'RMSE (Raiz do Erro Quadrático Médio): {rmse3:.2f}')